# Canadian Studies

---
BC

---

* load packages

In [ ]:
import warnings
from pathlib import Path

import geopandas as gpd
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray as rxr
from IPython.display import display

import RES.visuals as vis
from RES import utility as utils
from RES.hdf5_handler import DataHandler
import RES.lands as lands
# Setting plotting defaults to Elsevier style
plt.style.use('../RES/visual_styles/elsevier.mplstyle')
# Suppress specific warnings
warnings.filterwarnings("ignore", category=UserWarning)
cfg_BASELINE=utils.load_config('../config/config_CAN_baseline.yaml')
cfg_policy=utils.load_config('../config/config_CAN_policy1.yaml')

CRS_m = cfg_policy.get('region_mapping').get('BC').get('CRS_meters')  # Default metric CRS
CRS_d = cfg_policy.get('default_CRS').get('degrees')  # Default geographic CRS

sub_national_unit_tag=cfg_policy.get('GADM').get('datafield_mapping').get('NAME_2') # type: ignore

# Define Province Code

In [ ]:
# Construct region_options as a list of tuples: (name, code)
region_options = [(cfg_policy['region_mapping'][code]['name'], code) for code in cfg_policy['region_mapping']]
region_code = 'BC'  # Default selection, change as needed

# Create dropdown widget for region codes with names shown, codes as values
region_code_dropdown = widgets.Dropdown(
    options=region_options,
    value=region_code,
    description='Region:',
    disabled=False,
)

display(region_code_dropdown)

In [ ]:
region_code = region_code_dropdown.value
region_name=cfg_policy.get('region_mapping').get(region_code).get('name') 
utils.print_banner(f"Selected Region: {region_name} ({region_code})")

country_name=cfg_policy.get('country','Canada') 
country_kwd=country_name.replace(' ','')

##  Define run/scenario

In [ ]:
# Define the directory and search pattern
data_store_dir = Path("../data/store/")
search_keyword = f"resources_{country_kwd}_{region_code}_"

# List files containing the search pattern
matching_files = [str(f) for f in data_store_dir.glob(f"*{search_keyword}*") if f.is_file()]

# Extract run IDs from matching_files
POLICYs = [f.replace(str(data_store_dir) + '/', '').replace(search_keyword, '').replace(".h5", '') for f in matching_files]
utils.print_update(level=1,message=f"Results Available for Run IDs | {region_code}: ")
utils.print_update(level=2, message="\n    ".join(POLICYs))

# Load Validation data 

## Existing VRE sites

In [ ]:
existing_VREs_data_path=Path(f"../data/downloaded_data/CODERS/data-pull/supply/{region_code}_wind_generators.csv")
if existing_VREs_data_path.exists():
    existing_VREs=pd.read_csv(existing_VREs_data_path)
    existing_VREs_gdf=gpd.GeoDataFrame(existing_VREs,geometry=gpd.points_from_xy(existing_VREs.longitude,existing_VREs.latitude),crs=CRS_d)
    utils.print_update(level=1,message=f"Validation data for existing VREs loaded from {existing_VREs_data_path}")
else:
    existing_VREs_gdf=None
    utils.print_warning(f"Validation data for existing VREs not found at {existing_VREs_data_path}")

## Committed VRE Sites (BCH CFPs)

In [ ]:
committed_VREs_data_path=Path("../ROD_2024/BCH_CFP24.geojson")
if committed_VREs_data_path.exists():
    committed_VREs_gdf=gpd.read_file(committed_VREs_data_path)
    if committed_VREs_gdf.crs is None:
        committed_VREs_gdf.set_crs(CRS_m, allow_override=True, inplace=True)
    if committed_VREs_gdf.crs != CRS_m:
        committed_VREs_gdf.to_crs(CRS_m, inplace=True)
    utils.print_update(level=1,message=f"Committed VREs data loaded from {committed_VREs_data_path}")
else:
    committed_VREs_gdf=None
    utils.print_warning(f"Validation data for Committed VREs not found at {committed_VREs_data_path}")

# Load Data from Store

## Load Store

In [ ]:
store_all_runs={}
for POLICY in POLICYs:
    store=f"../data/store/resources_{country_kwd}_{region_code}_{POLICY}.h5"# f"../data/store/resources_{province_code}.h5" 
    res_data=DataHandler(store,show_structure=False) # the DataHandler object could be initiated without the store definition as well.
    store_all_runs[POLICY]=res_data

utils.print_update(level=1,message=f"Loaded store for {len(store_all_runs)} runs for {region_code}:")
utils.print_update(level=2,message="\n    ".join(store_all_runs.keys()))

- If Lines was not used/pulled 

In [ ]:
# lines=gpd.read_file('../data/downloaded_data/OSM/BC_power.geojson')
# # Select columns using boolean indexing
# columns_to_keep = ['power', 'voltage', 'geometry']
# lines_subset = lines.loc[:, lines.columns.isin(columns_to_keep)].copy()

# lines_subset["power"] = lines_subset["power"].astype(str)
# # Handle voltage conversion with error handling for non-numeric values
# lines_subset["voltage"] = pd.to_numeric(lines_subset["voltage"], errors='coerce').fillna(0).astype(int)

# res_data.to_store(lines_subset, 'lines',force_update=True)

- load dfs

In [ ]:
BC_dfs_all_runs={}

for POLICY, res_data in store_all_runs.items():
    utils.print_update(level=1,message=f"Loading dataframes for POLICY: {POLICY}")
    
    # Initialize dictionary for this POLICY
    BC_dfs_all_runs[POLICY] = {}
    BC_dfs_policy = BC_dfs_all_runs[POLICY]
    
    # Loading dataframes
    BC_dfs_policy['cells'] = res_data.from_store('cells')
    BC_dfs_policy['boundary'] = res_data.from_store('boundary')
    BC_dfs_policy['lines'] = res_data.from_store('lines')
    BC_dfs_policy['substations'] = res_data.from_store('substations')
    BC_dfs_policy['timeseries_solar'] = res_data.from_store('timeseries/solar')
    BC_dfs_policy['timeseries_wind'] = res_data.from_store('timeseries/wind')
    BC_dfs_policy['timeseries_clusters_solar'] = res_data.from_store('timeseries/clusters/solar')
    BC_dfs_policy['timeseries_clusters_wind'] = res_data.from_store('timeseries/clusters/wind')
    BC_dfs_policy['clusters_solar'] = res_data.from_store('clusters/solar')
    BC_dfs_policy['clusters_wind'] = res_data.from_store('clusters/wind')

- Load Boundary and Create Region Numbers for Plotting purposes

In [ ]:
boundary=BC_dfs_all_runs['BASELINE']['boundary']

region_mapping_data_path=Path(f'../data/region_mapping_{region_code}.csv')

if 'Region_number' not in boundary.columns:
    if region_mapping_data_path.exists():
        region_mapping=pd.read_csv(region_mapping_data_path)
        region_mapping['Region_Number'] = range(1, len(region_mapping) + 1)
     
        # Merge region names into boundary using region_mapping
        boundary = boundary.merge(region_mapping, left_on='Region', right_on='Region', how='left')
        # If any Region_Number is NaN, assign sequential numbers
    else:
        boundary['Region_Number'] = range(1, len(boundary) + 1)
        region_mapping = pd.DataFrame({
            'Region': boundary['Region'],
            'Region_Number': boundary['Region_Number']
        })
        
if boundary.crs is None:
    boundary.set_crs(CRS_d, allow_override=True, inplace=True)
if boundary.crs != CRS_m:
    boundary_proj=boundary.to_crs(CRS_m)

# Custom VIS Utils Funcs

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


def get_existing_committed_VRE_plot(
    ax: plt.Axes,
    target_crs: str = CRS_m,
    existing_VREs_gdf: gpd.GeoDataFrame = existing_VREs_gdf,
    committed_VREs_gdf: gpd.GeoDataFrame = committed_VREs_gdf,
    existing_VRE_type_column: str = "gen_type",
    existing_marker_col: str = "facility_installed_capacity",
    committed_marker_col: str = "potential_capacity",
    marker_scale_existing: float = 10.0,
    marker_scale_committed: float = 2.0,
    sites_legend_handle_scale: float = 8.0,
    marker_highlight_width: float = 5.0,
):
    # Reproject
    vre_proj = None
    # Existing VREs
    if existing_VREs_gdf is not None and not existing_VREs_gdf.empty:
        vre_proj = existing_VREs_gdf.to_crs(target_crs).copy()
        if existing_VRE_type_column not in vre_proj.columns:
            vre_proj[existing_VRE_type_column] = ""
        vre_proj[existing_VRE_type_column] = vre_proj[existing_VRE_type_column].fillna("").astype(str)

    # Committed VREs
    if committed_VREs_gdf is not None and not committed_VREs_gdf.empty:
        committed_proj = committed_VREs_gdf.to_crs(target_crs).copy()
        if existing_VRE_type_column not in committed_proj.columns:
            committed_proj[existing_VRE_type_column] = ""
        committed_proj[existing_VRE_type_column] = committed_proj[existing_VRE_type_column].fillna("").astype(str)

    legend_handles: list = []

    # Existing VREs
    if vre_proj is not None and not vre_proj.empty:
        sizes = pd.to_numeric(vre_proj.get(existing_marker_col, 1.0), errors="coerce").fillna(1.0) * marker_scale_existing

        is_wind = vre_proj[existing_VRE_type_column].str.lower().str.contains("wind", regex=False)
        is_solar = vre_proj[existing_VRE_type_column].str.lower().str.contains("solar", regex=False)

        if is_wind.any():
            vre_proj.loc[is_wind].plot(
                ax=ax, facecolor="None", edgecolor="blue",
                markersize=sizes[is_wind], marker="s", alpha=1, zorder=4,
                path_effects=[pe.withStroke(linewidth=marker_highlight_width, foreground="yellow", alpha=0.6)]
            )
            legend_handles.append(Line2D([0],[0], marker="s", color="blue", linestyle="None",
                                         markersize=8, markerfacecolor="None", label="Existing Wind"))

        if is_solar.any():
            vre_proj.loc[is_solar].plot(
                ax=ax, facecolor="None", edgecolor="red",
                markersize=sizes[is_solar], marker="s", alpha=1, zorder=4,
                path_effects=[pe.withStroke(linewidth=marker_highlight_width, foreground="yellow", alpha=0.6)]
            )
            legend_handles.append(Line2D([0],[0], marker="s", color="red", linestyle="None",
                                         markersize=8, markerfacecolor="None", label="Existing Solar"))

    # Committed VREs
    if committed_proj is not None and not committed_proj.empty:
        sizes_c = pd.to_numeric(committed_proj.get(committed_marker_col, 1.0), errors="coerce").fillna(1.0)
        sizes_c = sizes_c * marker_scale_committed

        is_wind_c = committed_proj["resource_type"].str.lower().str.contains("wind", regex=False)
        is_solar_c = committed_proj["resource_type"].str.lower().str.contains("solar", regex=False)

        if is_wind_c.any():
            committed_proj.loc[is_wind_c].plot(
                ax=ax, facecolor="None", edgecolor="fuchsia",
                markersize=sizes_c[is_wind_c], marker="^", alpha=1, zorder=5,
                path_effects=[pe.withStroke(linewidth=marker_highlight_width, foreground="yellow", alpha=0.6)]
            )
            legend_handles.append(Line2D([0],[0], marker="^", color="fuchsia", linestyle="None",
                                         markersize=sites_legend_handle_scale, markerfacecolor="None", label="Committed Wind"))

        if is_solar_c.any():
            committed_proj.loc[is_solar_c].plot(
                ax=ax, facecolor="None", edgecolor="coral",
                markersize=sizes_c[is_solar_c], marker="D", alpha=1, zorder=5,
                path_effects=[pe.withStroke(linewidth=marker_highlight_width, foreground="yellow", alpha=0.6)]
            )
            legend_handles.append(Line2D([0],[0], marker="D", color="coral", linestyle="None",
                                         markersize=sites_legend_handle_scale, markerfacecolor="None", label="Committed Solar"))

    return ax, legend_handles


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Mapping, Sequence

import geopandas as gpd
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import BoundaryNorm, ListedColormap, to_rgba
from matplotlib.patches import Patch


def plot_developable_land_and_vres(
    *,
    target_crs: str = CRS_m,
    raster_data: xr.DataArray,
    raster_legends: pd.DataFrame,
    classes_to_plot: Mapping[str, Sequence[int]] | None = None,  # {"solar":[...], "wind":[...]} or None -> ALL
    boundary: gpd.GeoDataFrame,
    existing_VREs_gdf: gpd.GeoDataFrame | None = None,   # expects `vre_type_column` and `existing_marker_col`
    include_tags: Iterable[str] = ("solar", "wind"),
    existing_marker_col: str = "wind_turbine_capacity",
    committed_VREs_gdf: gpd.GeoDataFrame | None = None,  # expects `vre_type_column` and `committed_marker_col`
    committed_marker_col: str = "potential_capacity",
    marker_scale_existing: float = 7.0,                           # multiplier for markersize columns   
    marker_scale_committed: float = 4.0,                           # multiplier for markersize columns
    area_labels: bool = False,
    title: str = "Developable Land with Existing & Committed VREs",
    fallback_crs: str = "EPSG:4326",
    label_column: str = "Country",
    vre_type_column: str = "gen_type",
    output_path: Path | str | None = None,
    figsize: tuple[float, float] = (12, 12),
    legend_anchor: tuple[float, float] | None = None,  # (x, y) in axis coordinates; None -> outside right
    dpi: int = 500,
    show: bool = True,
) -> tuple[plt.Figure, plt.Axes, Path | None]:

    # ---------- Legend helpers ----------
    id_to_name = dict(zip(raster_legends["class"].astype(int),
                          raster_legends["description"].astype(str)))
    id_to_hex  = dict(zip(raster_legends["class"].astype(int),
                          raster_legends["color"].astype(str)))

    # ---------- Decide and align CRS ----------
    if raster_data.rio.crs != target_crs:
        raster_plot = raster_data.rio.reproject(target_crs)   # <- fixed typo
    else:
        raster_plot = raster_data
    
    if boundary.crs != target_crs:
        boundary_proj = boundary.to_crs(target_crs)
    else:
        boundary_proj = boundary

    # ---------- Raster + classes logic ----------
    raster = raster_plot.values.squeeze().astype("int32", copy=False)
    codes_present = np.unique(raster)

    if classes_to_plot is None:
        selected_codes = set(int(c) for c in codes_present.tolist())
        selected_codes.discard(0)  # Assuming 0 is 'no data' or 'unclassified'
        use_masking = False
    else:
        selected_codes = set()
        for tag in include_tags:
            if tag in classes_to_plot:
                selected_codes.update(int(c) for c in classes_to_plot[tag])
        use_masking = True

    if use_masking:
        clc_masked = np.full_like(raster, -1, dtype="int32")
        if selected_codes:
            mask = np.isin(raster, list(selected_codes))
            clc_masked[mask] = raster[mask]
        values_to_color = sorted(selected_codes)
    else:
        clc_masked = raster
        values_to_color = sorted(set(int(v) for v in codes_present.tolist()))

    # ---------- Colormap & norm ----------
    color_map_dict: dict[int, tuple] = {}
    if use_masking:
        color_map_dict[-1] = (0, 0, 0, 0)   # transparent for excluded

    for c in values_to_color:
        color_map_dict[c] = to_rgba(id_to_hex.get(c, "#FFFFFF"))

    all_vals = sorted(color_map_dict.keys())
    cmap = ListedColormap([color_map_dict[v] for v in all_vals])
    last_edge = (all_vals[-1] + 1) if all_vals else 1
    norm = BoundaryNorm(np.array(all_vals + [last_edge]), cmap.N)

    # ---------- Bounds & extent ----------
    xmin, ymin, xmax, ymax = raster_plot.rio.bounds()
    x = raster_plot.coords["x"].values
    y = raster_plot.coords["y"].values
    extent = [float(x.min()), float(x.max()), float(y.min()), float(y.max())]
    origin = "upper" if y[0] > y[-1] else "lower"

    # ---------- Plot ----------
    fig, ax = plt.subplots(figsize=figsize)

    ax.imshow(
        clc_masked,
        extent=extent,
        origin=origin,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
        zorder=0,
    )

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Boundaries
    if not boundary_proj.empty:
        boundary_proj.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.5, zorder=3)

    # Labels
    if area_labels and label_column in boundary_proj.columns:
        for _, row in boundary_proj.iterrows():
            c = row.geometry.centroid
            ax.annotate(
                str(row[label_column]),
                (c.x, c.y),
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                color="black",
                path_effects=[pe.withStroke(linewidth=3, foreground="white")],
            )


    # ---------- VRE plotting ----------
    legend_handles: list[Patch] = []

    # ---------- Legend ----------
    # Class legend
    handles_classes: list[Patch] = []
    for c in values_to_color:
        base_label = f"{c}: {id_to_name.get(c, f'Class {c}')}"
        if classes_to_plot is None:
            label = base_label
        else:
            tag_note = []
            for tag in include_tags:
                if tag in classes_to_plot and c in set(int(v) for v in classes_to_plot[tag]):
                    tag_note.append(tag.capitalize())
            label = f"{base_label} ({' & '.join(tag_note)})" if tag_note else base_label

        handles_classes.append(
            Patch(facecolor=id_to_hex.get(c, "#888888"), edgecolor="none", label=label)
        )

    legend_all = handles_classes.copy()

    if use_masking:
        legend_all.append(Patch(facecolor="lightgrey", edgecolor="grey", hatch="///",
                                label="Excluded Lands (not shown)"))

    legend_all.extend(legend_handles)
    
    ax,VRE_legend_handles=get_existing_committed_VRE_plot(
        ax=ax,
        target_crs=target_crs,
        existing_VREs_gdf=existing_VREs_gdf,
        committed_VREs_gdf=committed_VREs_gdf,
        existing_VRE_type_column=vre_type_column,
        existing_marker_col=existing_marker_col,
        committed_marker_col=committed_marker_col,
        marker_scale_existing=marker_scale_existing,
        marker_scale_committed=marker_scale_committed,
    )
    legend_all.extend(VRE_legend_handles)
    
    if legend_all:
        ax.legend(
            handles=legend_all,
            loc="upper left",            # anchor point inside the legend box
            bbox_to_anchor=legend_anchor if legend_anchor is not None else (1.02, 1), # (x, y) in axis coordinates
            fontsize=12,
            frameon=False,
        )

    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()

    saved: Path | None = None
    if output_path is not None:
        saved = Path(output_path)
        saved.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(saved, dpi=dpi, bbox_inches="tight")
        utils.print_update(level=1,message=f"Unique land-cover classes in the area: {saved}")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, ax, saved


# Compare Regional Capacity with Default Scenario

> Run the default scenario results first to set the benchmark

In [ ]:
utils.print_update(level=1,message=" Available run ids: ")
utils.print_update(level=2,message="\n    ".join(BC_dfs_all_runs.keys()))

- Define POLICY from the available results

In [ ]:
POLICY='strict_policy_aeroway_CPCAD_buffer'
BASELINE_results_save_to = utils.ensure_path(f'../results/{country_kwd}/{region_code}/BASELINE/')
POLICY_results_save_to = utils.ensure_path(f'../results/{country_kwd}/{region_code}/{POLICY}/')
BASELINE_vis_save_to = utils.ensure_path(f'../vis/{country_kwd}/{region_code}/BASELINE/')
POLICY_vis_save_to = utils.ensure_path(f'../vis/{country_kwd}/{region_code}/{POLICY}/')

In [ ]:
cells_scenario:pd.DataFrame=BC_dfs_all_runs[f'{POLICY}']['cells']
cells_baseline:pd.DataFrame=BC_dfs_all_runs['BASELINE']['cells']

if cells_scenario.crs is None:
    cells_scenario.set_crs(CRS_d, allow_override=True, inplace=True)
if cells_scenario.crs != CRS_m:
    cells_scenario_proj=cells_scenario.to_crs(CRS_m)

if cells_baseline.crs is None:  
    cells_baseline.set_crs(CRS_d, allow_override=True, inplace=True)
if cells_baseline.crs != CRS_m:
    cells_baseline_proj=cells_baseline.to_crs(CRS_m)

In [ ]:
from RES.CellCapacityProcessor import get_sub_nationally_aggregated_capacity

file_path_BASELINE=Path(f'{BASELINE_results_save_to}/cells_aggregated_by_{sub_national_unit_tag}_{region_code}_BASELINE.csv')
file_path_scenario=Path(f'{POLICY_results_save_to}/cells_aggregated_by_{sub_national_unit_tag}_{region_code}_{POLICY}.csv')


utils.print_update(level=2,
                        message=f"Calculating aggregated capacity for {sub_national_unit_tag} | BASELINE...")

# calculate for BASELINE  
cells_aggrs_region_BASELINE=get_sub_nationally_aggregated_capacity(cells_baseline,sub_national_unit_tag)
cells_aggrs_region_BASELINE.to_csv(file_path_BASELINE)
            
utils.print_update(level=3,
                        message=f"Aggregated cells by'{sub_national_unit_tag}' saved to '{file_path_BASELINE}'")


utils.print_update(level=2,
                        message=f"Calculating aggregated capacity for {sub_national_unit_tag} | {POLICY}...")
# Calculate for POLICY
cells_aggrs_region_scenario=get_sub_nationally_aggregated_capacity(cells_scenario,sub_national_unit_tag)
cells_aggrs_region_scenario.to_csv(file_path_scenario)

        
utils.print_update(level=3,
                        message=f"Aggregated cells by'{sub_national_unit_tag}' saved to '{file_path_scenario}'")

- Calculate Capacity variations

In [ ]:
delta_cells_aggrs_region=abs(cells_aggrs_region_BASELINE-cells_aggrs_region_scenario)

if delta_cells_aggrs_region.empty:
    utils.print_warning("No differences in aggregated capacities between BASELINE and POLICY scenarios.")
else:
    utils.print_update(level=1,message=f"Differences in aggregated capacities between BASELINE and {POLICY} scenarios:")
    display(delta_cells_aggrs_region.sum()/1E3,"in GW")

- Spatial- mapping the delta of capacity to Regions

In [ ]:
if (delta_cells_aggrs_region.sum(axis=0) == 0).all():
    utils.print_update(level=2,
                      message=f"No differences found between default and scenario aggregated cells for {sub_national_unit_tag}.")
    utils.print_warning("Perhaps we are looking at the same scenario?")
else:
    delta_cells_aggrs_region_gdf=boundary.copy()
    
    # - For plotting, Update Boundary (gdf) with this delta_Capacity
    delta_cells_aggrs_region_gdf["potential_capacity_solar"] = delta_cells_aggrs_region_gdf['Region'].str.replace(' ', '').map(delta_cells_aggrs_region["potential_capacity_solar"])
    delta_cells_aggrs_region_gdf["potential_capacity_wind"] = delta_cells_aggrs_region_gdf['Region'].str.replace(' ', '').map(delta_cells_aggrs_region["potential_capacity_wind"])

    # Map Population and GDP to gdf using Region
    delta_cells_aggrs_region_gdf["potential_capacity_solar_GW"] = delta_cells_aggrs_region_gdf["potential_capacity_solar"].apply(lambda x: x / 1E3) # GW
    delta_cells_aggrs_region_gdf["potential_capacity_wind_GW"] = delta_cells_aggrs_region_gdf["potential_capacity_wind"].apply(lambda x: x / 1E3) # GW

- plot in map

In [ ]:
# Plot configuration 
plot_config = { 'solar': {'column': 'potential_capacity_solar_GW', 'cmap': 'YlOrRd', 'label': 'Solar potential (GW)'}, 'wind': {'column': 'potential_capacity_wind_GW', 'cmap': 'BuPu', 'label': 'Wind potential (GW)'} }

In [ ]:
import matplotlib.patheffects as pe


# --- Helper: Plot with shadow + top 5 highlights ---
def plot_map_with_shadow(ax, delta_capacity_gdf, column, cmap, shadow_offset=0.0001, top_n=12):
    """Plot map with shadow effect and highlight top-N rows."""
    
    # Ensure CRS consistency
    if delta_capacity_gdf.crs != CRS_m:
        delta_capacity_gdf = delta_capacity_gdf.to_crs(CRS_m)

    # Shadow
    shadow_geom = delta_capacity_gdf.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
    gpd.GeoDataFrame(geometry=shadow_geom, crs=CRS_m).plot(
        ax=ax, facecolor='none', edgecolor='gray', linewidth=1.3, alpha=0.3
    )

    # Main plot
    delta_capacity_gdf.plot(column=column, ax=ax, cmap=cmap, edgecolor='black', linewidth=0.2, legend=False)

    # --- Highlight top N values ---
    top_gdf = delta_capacity_gdf.nlargest(top_n, column)
    for _, row in top_gdf.iterrows():
        if row[column] > 0:
            x, y = row.geometry.centroid.x, row.geometry.centroid.y
            ax.text(x, y, f"{row[column]:.1f}",
                    ha="center", va="center", fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=1.1, foreground="k", alpha=0.6)])

    # Return scalar mappable for colorbar
    return plt.cm.ScalarMappable(
        cmap=cmap,
        norm=plt.Normalize(vmin=delta_capacity_gdf[column].min(),
                           vmax=delta_capacity_gdf[column].max())
    )

# --- Create plots ---
fig, axes = plt.subplots(dpi=1000, ncols=2, figsize=(8, 3),constrained_layout=True)
fig.suptitle("Potential capacity lost due to landuse policy implication",
             weight='bold', fontsize=12,y=1)
plt.subplots_adjust(wspace=0.001)  # reduce horizontal space between subplots
for ax in axes:
    ax.set_axis_off()

for ax, (key, config) in zip(axes, plot_config.items()):
    sm = plot_map_with_shadow(ax, delta_cells_aggrs_region_gdf,
                              config['column'], config['cmap'], top_n=10)
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6)
    cbar.set_label(config['label'], fontsize=12)

# plt.tight_layout()
plt.savefig(f'{POLICY_vis_save_to}/potential_capacity_lost_default_vs_{POLICY}.svg',
            bbox_inches='tight')

#DOC content
plt.savefig('../docs/source/_static/potential_capacity_lost_default_vs_policy_aeroway_CPCAD_buffer.png')


# Lands

## Extract Boundary atttributes
> To clip the rasters, plotting etc.


In [ ]:
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = boundary.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}

In [ ]:
landcover_raster_path = f"../data/downloaded_data/GAEZ/Rasters_in_use/LR/lco/{region_code}_faocmb_2010.tif"
landcover_raster_legends=pd.read_csv("../data/faocmb_2010_legend.csv")
land_cover_cfg= next((item for item in cfg_policy.get('GAEZ').get('raster_types') if 'land_cover' in item.get('name', '')), None)
layers:dict=land_cover_cfg['class_inclusion']
landcover_raster_data = (
        rxr.open_rasterio(landcover_raster_path)
        .rio.clip_box(**bounding_box_dict)
    )

# Unique CLC codes in your clipped area
unique_classes = np.unique(landcover_raster_data.values[~np.isnan(landcover_raster_data.values)])
fig1,ax1,save_to1=plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=landcover_raster_data,
    raster_legends=landcover_raster_legends,
    classes_to_plot=layers, #layers,                 # <- all classes
    boundary=boundary,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    marker_scale_existing=10.0,
    marker_scale_committed=1.0,
    title="Suitable Landcovers (GAEZ_v4 FAO-CMB 2010)",
    existing_marker_col="wind_turbine_capacity",
    output_path=f"../vis/{country_kwd}/{region_code}/Landcover_with_existing_VREs.svg",
    legend_anchor=(.73, 1)
)

# Update DOC contents (if needed)
doc_save_to1="../docs/source/_static/Landcover_with_existing_VREs.svg"
fig1.savefig(doc_save_to1, bbox_inches="tight")
utils.print_update(level=1,message=f"plot saved to:: {doc_save_to1}")

fig1.savefig(f"{POLICY_vis_save_to}/Landcover_with_existing_VREs.svg", bbox_inches="tight")
utils.print_update(level=1,message=f"plot saved to: {POLICY_vis_save_to}/Landcover_with_existing_VREs.svg")

In [ ]:
terrain_raster_path = f"../data/downloaded_data/GAEZ/Rasters_in_use/LR/ter/{region_code}_slpmed05.tif"
terrain_raster_legends=pd.read_csv("../data/gaez_slpmed05_legend.csv")
land_cover_cfg= next((item for item in cfg_policy.get('GAEZ').get('raster_types') if 'terrain_resources' in item.get('name', '')), None)
layers_excluded:dict=land_cover_cfg['class_exclusion']

terrain_raster_data = (
        rxr.open_rasterio(terrain_raster_path)
        .rio.clip_box(**bounding_box_dict)
    )

# Unique CLC codes in your clipped area
unique_classes = np.unique(terrain_raster_data.values[~np.isnan(terrain_raster_data.values)])

layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}


fig2,ax2,save_to2=plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=terrain_raster_data,
    raster_legends=terrain_raster_legends,
    classes_to_plot=layers_included, #layers,                 # <- all classes
    boundary=boundary,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
        marker_scale_existing=10.0,
    marker_scale_committed=1.0,
    title="Terrains (GAEZ_v4 slop median 0.5)",
    existing_marker_col="wind_turbine_capacity",
    output_path=f"../vis/{country_kwd}/{region_code}/terrains_with_existing_VREs.svg",
    legend_anchor=(.73, 1)
)

# Update DOC contents (if needed)
doc_save_to2="../docs/source/_static/terrains_with_existing_VREs.svg"
fig2.savefig(doc_save_to2, bbox_inches="tight")
utils.print_update(level=1,message=f"plot saved to: {doc_save_to2}")
fig2.savefig(f"{POLICY_vis_save_to}/terrains_with_existing_VREs.svg", bbox_inches="tight")
utils.print_update(level=1,message=f"plot saved to: {POLICY_vis_save_to}/terrains_with_existing_VREs.png")

In [ ]:
excld_raster_path = f"../data/downloaded_data/GAEZ/Rasters_in_use/LR/excl/{region_code}_exclusion_2017.tif"
excld_raster_legends=pd.read_csv("../data/exclusion_2017_legend.csv")
excld_cover_cfg= next((item for item in cfg_policy.get('GAEZ').get('raster_types') if 'exclusion_areas' in item.get('name', '')), None)
layers_excluded:dict=excld_cover_cfg['class_exclusion']

excld_raster_data = (
        rxr.open_rasterio(excld_raster_path)
        .rio.clip_box(**bounding_box_dict)
    )

# Unique CLC codes in your clipped area
unique_classes = np.unique(excld_raster_data.values[~np.isnan(excld_raster_data.values)])
layers_included = {
    tech: [c for c in unique_classes if c not in layers_excluded[tech] and c != 0]
    for tech in ['solar', 'wind']
}

fig3,ax3,save_to3=plot_developable_land_and_vres(
    target_crs=CRS_m,
    raster_data=excld_raster_data,
    raster_legends=excld_raster_legends,
    classes_to_plot=layers_excluded, #layers_excluded, #None,                 # <- all classes
    boundary=boundary,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
        marker_scale_existing=10.0,
    marker_scale_committed=1.0,
    title="Globally Protected Areas (GAEZ_v4 exclusion 2017)",
    existing_marker_col="wind_turbine_capacity",
    output_path=f"../vis/{country_kwd}/{region_code}/exclusion_global_with_existing_VREs.svg",
    legend_anchor=(.73, 1)
)
utils.print_update(level=1,message=f"Unique land-cover classes in the area: {save_to2}")

# Update DOC contents (if needed)
doc_save_to3="../docs/source/_static/exclusion_global_with_existing_VREs.svg"
fig3.savefig(doc_save_to3, bbox_inches="tight")
utils.print_update(level=1,message=f"plot saved to: {doc_save_to3}")

fig3.savefig(f"{POLICY_vis_save_to}/exclusion_global_with_existing_VREs.svg", bbox_inches="tight")
utils.print_update(level=1,message=f"plot saved to: {POLICY_vis_save_to}/exclusion_global_with_existing_VREs.png")

## Aeroway

In [ ]:
aeroway=gpd.read_file(f'../data/downloaded_data/OSM/{region_code}_aeroway.geojson')
if aeroway.crs is None:
    aeroway.set_crs(CRS_m, allow_override=True, inplace=True)
if aeroway.crs != CRS_m:
    aeroway.to_crs(CRS_m, inplace=True)

* Load_policy_config

In [ ]:
cfg_for_plot=cfg_BASELINE

In [ ]:
aeroway_buffer_solar:list=cfg_for_plot.get('capacity_disaggregation').get('solar').get('vector_buffers')
aeroway_solar:dict= next(
    (item for item in aeroway_buffer_solar if item.get("aeroway")),
    None
)

aeroway_buffer_wind:list=cfg_for_plot.get('capacity_disaggregation').get('wind').get('vector_buffers')
aeroway_wind:dict= next(
    (item for item in aeroway_buffer_wind if item.get("aeroway")),
    None
)

In [ ]:
aeroway_buffer_solar_gdf,B1=lands.apply_buffer_to_vector(aeroway,CRS_m,CRS_d,aeroway_solar['aeroway']['buffer_mapping_key_buffers'],
                                 aeroway_solar['aeroway']['buffer_mapping_key'])


aeroway_buffer_wind_gdf,B2=lands.apply_buffer_to_vector(aeroway,CRS_m,CRS_d,aeroway_wind['aeroway']['buffer_mapping_key_buffers'],
                                 aeroway_wind['aeroway']['buffer_mapping_key'])

In [ ]:
if boundary.crs != CRS_m:
    boundary_proj=boundary.to_crs(CRS_m)
else:
    boundary_proj=boundary

if aeroway_buffer_solar_gdf.crs != CRS_m:
    aeroway_buffer_solar_gdf_proj=aeroway_buffer_solar_gdf.to_crs(CRS_m)
    aeroway_buffer_wind_gdf_proj=aeroway_buffer_wind_gdf.to_crs(CRS_m)
else:
    aeroway_buffer_solar_gdf_proj=aeroway_buffer_solar_gdf
    aeroway_buffer_wind_gdf_proj=aeroway_buffer_wind_gdf


In [ ]:

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

# --- Collect all categories from both datasets ---
all_cats = pd.Index(
    pd.concat([
        aeroway_buffer_solar_gdf_proj["aeroway"],
        aeroway_buffer_wind_gdf_proj["aeroway"]
    ])
    .dropna()
    .unique()
)

# Build consistent color mapping
cmap =plt.get_cmap("prism", len(all_cats))
cat2color = {cat: cmap(i) for i, cat in enumerate(all_cats)}

# Map colors to each GeoDataFrame
aeroway_buffer_solar_gdf_proj["color"] = aeroway_buffer_solar_gdf_proj["aeroway"].map(cat2color)
aeroway_buffer_wind_gdf_proj["color"]  = aeroway_buffer_wind_gdf_proj["aeroway"].map(cat2color)

# --- Plot ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.5, 4), dpi=500)
fig.suptitle(f"Aeroway Buffers (BASELINE)", weight='bold', fontsize=12,y=0.93)
# Solar buffers
boundary_proj.plot(ax=ax1, facecolor="grey", edgecolor="k", linewidth=0.4, alpha=0.1, zorder=3)
aeroway_buffer_solar_gdf_proj.centroid.plot(
    ax=ax1,
    color=aeroway_buffer_solar_gdf_proj["color"],
    edgecolor='None',
    markersize=aeroway_buffer_solar_gdf_proj["buffer_applied_m"]/300,
    alpha=0.4
)
ax1.set_title("No-go Buffers (Solar)",fontweight='bold',fontsize=10,y=0.95)
ax1.axis("off")

# Wind buffers
boundary_proj.plot(ax=ax2, facecolor="grey", edgecolor="k", linewidth=0.4, alpha=0.1, zorder=3)
aeroway_buffer_wind_gdf_proj.centroid.plot(
    ax=ax2,
    color=aeroway_buffer_wind_gdf_proj["color"],
    edgecolor='None',
    markersize=aeroway_buffer_wind_gdf_proj["buffer_applied_m"]/300,
    alpha=0.3
)
ax2.set_title("No-go Buffers (Wind)",fontweight='bold',fontsize=10,y=0.94)
ax2.axis("off")

# --- Shared Legend ---
handles = [
    mpatches.Patch(color=col, label=lab) for lab, col in cat2color.items()
]
fig.legend(handles=handles, title="Aeroway Category",
           loc="lower center", ncol=6, frameon=False, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()
fig.savefig(f'../vis/{country_kwd}/{region_code}/aeroway_buffers.svg', bbox_inches='tight')

save_to_root=f'{BASELINE_vis_save_to}' if cfg_for_plot['Scenario']['run_id']=='BASELINE' else f'{POLICY_vis_save_to}'
# Doc content
fig.savefig('../docs/source/_static/aeroway_buffers.svg', bbox_inches='tight')
fig.savefig(f'{save_to_root}/aeroway_buffers.svg', bbox_inches='tight')

## CPCAD

In [ ]:
cpcad=pd.read_pickle(f'../data/downloaded_data/lands/ProtectedConservedArea_{region_code}.pickle')
if cpcad.crs is None:
    cpcad.set_crs(CRS_m, allow_override=True, inplace=True)
if cpcad.crs != CRS_m:
    cpcad.to_crs(CRS_m, inplace=True)

- Temp Fix (one time)

In [ ]:
# cpcad.to_pickle(f'../data/downloaded_data/lands/ProtectedConservedArea_{region_code}.pickle')
# # temp fix. New workflow should have this fixed.
# cpcad['IUCN_CAT_desc'] = cpcad['IUCN_CAT_desc'].replace(
#     {'Strict  Nature Reserve': 'Strict Nature Reserve'}
# )

In [ ]:
color_df = pd.read_csv("../data/CPCAD_legends.csv")
cat2color = dict(zip(color_df["IUCN_CAT_desc"], color_df["color_hex"]))

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

# Shadow offset for boundary
shadow_offset = 0.008

fig, ax = plt.subplots(figsize=(10,7), dpi=1000)

# ---- 1) Build color column from cat2color safely ----
# Assume cat2color = {"Wilderness Area": "#1f77b4", "National Park": "#ff7f0e", ...}
cpcad["color"] = cpcad["IUCN_CAT_desc"].map(cat2color)

# Handle unmapped categories (assign gray fallback)
cpcad["color"] = cpcad["color"].fillna("#d3d3d3")

# Keep a stable order for legend
ordered_cats = pd.Index(cpcad["IUCN_CAT_desc"].dropna().unique()).tolist()

# ---- 2) CPCAD polygons with shadow ----
cpcad_shadow = cpcad.copy()
cpcad_shadow.geometry = cpcad_shadow.geometry.translate(
    xoff=-shadow_offset, yoff=shadow_offset
)
cpcad_shadow.plot(ax=ax, color=cpcad_shadow["color"], linewidth=0, alpha=0.35)

cpcad.plot(ax=ax, color=cpcad["color"], linewidth=0)

# ---- 3) VRE layers ----
ax, VRE_legend_handles = get_existing_committed_VRE_plot(
    ax=ax,
    target_crs=CRS_m,
    existing_VREs_gdf=existing_VREs_gdf,
    committed_VREs_gdf=committed_VREs_gdf,
    marker_scale_existing=0.1,
    marker_scale_committed=0.25,
    sites_legend_handle_scale=6.0,
    marker_highlight_width=5.0
)

# ---- 4) Boundary with shadow ----
boundary_proj = boundary.to_crs(CRS_m) if boundary.crs != CRS_m else boundary
boundary_proj.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.3, alpha=1)
boundary_proj_shadow = boundary_proj.copy()
boundary_proj_shadow.geometry = boundary_proj_shadow.geometry.translate(
    xoff=shadow_offset, yoff=-shadow_offset
)
boundary_proj_shadow.plot(ax=ax, facecolor="none", edgecolor="gray", linewidth=0.2, alpha=0.9)

# ---- 5) Legend handles from cat2color (with fallback) ----
cpcad_handles = [
    mpatches.Patch(facecolor=cat2color.get(lab, "#d3d3d3"),
                   edgecolor="none", label=lab)
    for lab in ordered_cats
]

all_handles = cpcad_handles + VRE_legend_handles
ax.legend(
    handles=all_handles,
    # title="Protected Areas & VRE Projects",
    loc="upper left",
    bbox_to_anchor=(0.7, .9),
    frameon=False,
    prop={"size": 11}
)

# ---- 6) Final touches ----
ax.grid(False)
ax.axis("off")
plt.tight_layout()

plt.savefig(f"../docs/source/_static/CPCAD_{region_code}.png", bbox_inches="tight")
plt.savefig(f"../vis/{country_kwd}/{region_code}/CPCAD_{region_code}.png", bbox_inches="tight")

- Visualize scenario Buffers

In [ ]:
cfg_for_plot=cfg_BASELINE

In [ ]:
cpcad_buffer_solar:list=cfg_for_plot.get('capacity_disaggregation').get('solar').get('vector_buffers')
cpcad_solar:dict= next(
    (item for item in cpcad_buffer_solar if item.get("conserved_lands")),
    None
)

cpcad_buffer_wind:list=cfg_for_plot.get('capacity_disaggregation').get('wind').get('vector_buffers')
cpcad_wind:dict= next(
    (item for item in cpcad_buffer_wind if item.get("conserved_lands")),
    None
)

In [ ]:
cpcad_buffer_solar_gdf,B1=lands.apply_buffer_to_vector(cpcad,CRS_m,CRS_d,cpcad_solar['conserved_lands']['buffer_mapping_key_buffers'],cpcad_solar['conserved_lands']['buffer_mapping_key'])

cpcad_buffer_wind_gdf,B2=lands.apply_buffer_to_vector(cpcad,CRS_m,CRS_d,cpcad_wind['conserved_lands']['buffer_mapping_key_buffers'],cpcad_wind['conserved_lands']['buffer_mapping_key'])

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd

# Shadow offset for boundary
shadow_offset = 0.008

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.5, 5), dpi=1000)
fig.suptitle(f"Canadian Protected and Conserved Areas (2023) [BASELINE]",
             fontsize=16, fontweight='bold', y=0.95)

# --- Ensure consistent CRS ---
cpcad_buf_solar = (cpcad_buffer_solar_gdf.to_crs(CRS_m)
                   if cpcad_buffer_solar_gdf.crs != CRS_m else cpcad_buffer_solar_gdf).copy()
cpcad_buf_wind  = (cpcad_buffer_wind_gdf.to_crs(CRS_m)
                   if cpcad_buffer_wind_gdf.crs != CRS_m else cpcad_buffer_wind_gdf).copy()
boundary_proj   = boundary.to_crs(CRS_m) if boundary.crs != CRS_m else boundary

# --- Apply cat2color mapping ---
# Load from CSV or define manually:
# color_df = pd.read_csv("CPCAD_color_codes.csv")
# cat2color = dict(zip(color_df["IUCN_CAT_desc"], color_df["color_hex"]))

cpcad_buf_solar["color"] = cpcad_buf_solar["IUCN_CAT_desc"].map(cat2color).fillna("#d3d3d3")
cpcad_buf_wind["color"]  = cpcad_buf_wind["IUCN_CAT_desc"].map(cat2color).fillna("#d3d3d3")

# --- Left: Solar buffer ---
cpcad_buf_solar.plot(color=cpcad_buf_solar["color"], ax=ax1, edgecolor="none")
boundary_proj.plot(ax=ax1, facecolor='none', edgecolor='gray', linewidth=0.3, alpha=1)
boundary_proj.translate(xoff=shadow_offset, yoff=-shadow_offset).plot(
    ax=ax1, facecolor='none', edgecolor='gray', linewidth=0.2, alpha=0.9
)
ax1.set_title("No-go Buffers (Solar)", fontsize=14, y=0.93)
ax1.axis('off')

# --- Right: Wind buffer ---
cpcad_buf_wind.plot(color=cpcad_buf_wind["color"], ax=ax2, edgecolor="none")
boundary_proj.plot(ax=ax2, facecolor='none', edgecolor='gray', linewidth=0.3, alpha=1)
boundary_proj.translate(xoff=shadow_offset, yoff=-shadow_offset).plot(
    ax=ax2, facecolor='none', edgecolor='gray', linewidth=0.2, alpha=0.9
)
ax2.set_title("No-go Buffers (Wind)", fontsize=14, y=0.93)
ax2.axis('off')

# --- Legend (from cat2color, ensures consistency) ---
handles = [mpatches.Patch(facecolor=color, edgecolor="none", label=cat)
           for cat, color in cat2color.items()]
fig.legend(handles=handles, title="IUCN Category",
           loc="lower center", ncol=3, frameon=False, fontsize=10)

# --- Layout tidy ---
plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.savefig(f'../docs/source/_static/CPCAD_{region_code}_buffers.png', bbox_inches="tight")

save_to_root=f'{BASELINE_vis_save_to}' if cfg_for_plot['Scenario']['run_id']=='BASELINE' else f'{POLICY_vis_save_to}'
plt.savefig(f'{save_to_root}/CPCAD_buffers_side_by_side_{region_code}.svg', bbox_inches="tight")

# Clusters

In [ ]:
clusters_wind=BC_dfs_all_runs[f'{POLICY}']['clusters_wind']
clusters_solar=BC_dfs_all_runs[f'{POLICY}']['clusters_solar']

In [ ]:
clusters_wind_f=clusters_wind[clusters_wind['potential_capacity']>0]
clusters_solar_f=clusters_solar[clusters_solar['potential_capacity']>0]

In [ ]:
print(f'Total sites {len(clusters_wind_f)}')
total_capacity=clusters_wind_f.potential_capacity.sum()
print(f'Total Capacity {int(total_capacity/1E3)} GW')
sites=5
top_sites_capacity=clusters_wind_f.head(sites).potential_capacity.sum()
print(f'Top {sites} sites ({round(sites/len(clusters_wind_f)*100)}% site) capacity {int(top_sites_capacity/1E3)} GW ({round(top_sites_capacity/total_capacity*100)}% of total capacity)')

In [ ]:
print(f'Total sites {len(clusters_solar_f)}')
total_capacity=clusters_solar_f.potential_capacity.sum()
print(f'Total Capacity {int(total_capacity/1E3)} GW')
sites=5
top_sites_capacity=clusters_solar_f.head(sites).potential_capacity.sum()
print(f'Top {sites} sites ({round(sites/len(clusters_solar_f)*100)}% site) capacity {int(top_sites_capacity/1E3)} GW ({round(top_sites_capacity/total_capacity*100)}% of total capacity)')

In [ ]:
clusters_wind_f=clusters_wind[clusters_wind['potential_capacity']>0]
clusters_solar_f=clusters_solar[clusters_solar['potential_capacity']>0]

## Cluster-Timeseries


In [ ]:
dissolved_indices_solar=store_all_runs[f"{POLICY}"].from_store('dissolved_indices/solar')
cell_ts_solar=store_all_runs[f"{POLICY}"].from_store('timeseries/solar')
dissolved_indices_wind=store_all_runs[f"{POLICY}"].from_store('dissolved_indices/wind')
cell_ts_wind=store_all_runs[f"{POLICY}"].from_store('timeseries/wind')

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def get_cluster_vs_cells_profile(region_to_plot: str, 
                                 cluster_id_to_plot: int,
                                 resource_type:str,
                                 cells_timeseries: pd.DataFrame, 
                                 dissolved_indices: pd.DataFrame, 
                                 cluster_timeseries: pd.DataFrame,
                                 plot_save_to: str | Path = None):
    """
    Plot cluster representative profile vs all member cells as daily mean ± std deviation.

    Parameters
    ----------
    region_to_plot : str
        Region name.
    cluster_id_to_plot : int
        Cluster ID.
    cells_timeseries : pd.DataFrame
        Timeseries for all cells, indexed by datetime.
    dissolved_indices : pd.DataFrame
        Mapping of cells to clusters: dissolved_indices.loc[region, cluster_id] gives list of cell IDs.
    cluster_timeseries : pd.DataFrame
        Cluster representative timeseries, indexed by datetime.
    plot_save_to : str | Path, optional
        Path to save figure. If None, figure is saved under `./vis/`.
    """
    resource_type = resource_type.lower()
    if resource_type not in ['solar', 'wind']:
        raise ValueError("resource_type must be either 'solar' or 'wind'")
    region = region_to_plot
    cluster_id = cluster_id_to_plot

    # --- 1. GET MEMBER CELLS OF THE CLUSTER ---
    cell_ids = dissolved_indices.loc[region, cluster_id]

    # --- 2. RESAMPLE TO DAILY MEAN ---
    cell_daily = [cells_timeseries[cid].resample("1D").mean() for cid in cell_ids]
    cluster_daily = cluster_timeseries[f"{region}_{cluster_id}"].resample("1D").mean()

    # --- 3. CONVERT TO 2D ARRAY (days × cells) ---
    cell_matrix = np.column_stack([s.values for s in cell_daily])

    # --- 4. CALCULATE DAILY MEAN AND STD DEV ---
    mean_cells = cell_matrix.mean(axis=1)
    std_cells = cell_matrix.std(axis=1)

    # --- 5. PLOT ---
    fig, ax = plt.subplots(figsize=(12, 3.5), dpi=1000)

    # Shaded area: ±1 std deviation
    ax.fill_between(cell_daily[0].index, mean_cells - std_cells, mean_cells + std_cells,
                    color="orange" if resource_type=='solar' else "skyblue", alpha=0.3, label="Cells ±1 Std Dev")

    # Cluster representative
    ax.plot(cluster_daily.index, cluster_daily.values, color="orangered" if resource_type=='solar' else "navy", linewidth=2, label="Cluster Profile")

    # Clean aesthetics
    ax.set_title(f"Daily Mean {resource_type.capitalize()} Profiles – {region} Cluster {cluster_id}", fontsize=14, weight="bold")
    # ax.set_xlabel("Day of Year", fontsize=12)
    ax.set_ylabel("Normalized Generation", fontsize=12)
    ax.grid(alpha=0.4,linestyle='--', linewidth=0.5)
    ax.legend(frameon=False,fontsize=13)
    ax.tick_params(axis="both", which="major", labelsize=10,labelrotation =90,direction='in',length=3)

    # Optional: make background transparent
    # fig.patch.set_alpha(0)
    # ax.set_facecolor('none')

    plt.tight_layout()

    # --- 6. SAVE FIGURE ---
    plot_save_to = Path(f'{POLICY_vis_save_to}/{resource_type}_cluster_{region}_{cluster_id}_vs_cells_profile.svg')
    plot_save_to.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(plot_save_to)
    utils.print_update(level=2, message=f"Cluster vs Cells profile plot saved to {plot_save_to}")
    plt.show()

In [ ]:
timeseries_clusters_solar=BC_dfs_all_runs[f'{POLICY}']['timeseries_clusters_solar']
timeseries_clusters_wind=BC_dfs_all_runs[f'{POLICY}']['timeseries_clusters_wind']

In [ ]:
get_cluster_vs_cells_profile('EastKootenay', 1, 'solar',cell_ts_solar, dissolved_indices_solar, timeseries_clusters_solar)
get_cluster_vs_cells_profile('PeaceRiver', 1, 'wind',cell_ts_wind, dissolved_indices_wind, timeseries_clusters_wind)

In [ ]:
# resource_clusters_solar,cluster_timeseries_solar=Builder.select_top_sites(solar_clusters,
#                                                                 solar_clusters_ts,
#                                                                     resource_max_capacity=10)

# resource_clusters_wind,cluster_timeseries_wind=RES_module.select_top_sites(wind_clusters,
#                                                                 wind_clusters_ts,
#                                                                     resource_max_capacity=30)

In [ ]:
# import matplotlib.pyplot as plt
# import geopandas as gpd
# import pandas as pd

# legend_x_ax_offset=1

# # Ensure 'Region' is in the columns for both boundary and cells
# if 'Region' not in boundary.columns:
#     boundary = boundary.reset_index(inplace=True)

# # Assign a number to each region
# boundary['Region_Number'] = range(1, len(boundary) + 1)

# # Define custom bins and labels for solar and wind capacity
# solar_bins = [0, 100, 200, 300, 500, float('inf')]  # Custom ranges
# solar_labels = ['<100','100-200', '200-300', '300-500','>500']  # Labels for legend

# # Define custom bins and labels for solar and wind capacity
# wind_bins = [0, 300, 500, 1000, 2000,3000, float('inf')]  # Custom ranges
# wind_labels = ['<300','300-500', '500-1000', '1000-2000','2000-3000', '>3000']  # Labels for legend

# # Categorize potential_capacity_solar and potential_capacity_wind into bins
# resource_clusters_solar['solar_category'] = pd.cut(resource_clusters_solar['potential_capacity'], bins=solar_bins, labels=solar_labels, include_lowest=True)
# resource_clusters_wind['wind_category'] = pd.cut(resource_clusters_wind['potential_capacity'], bins=wind_bins, labels=wind_labels, include_lowest=True)

# # Create figure and axes for side-by-side plotting
# fig, (ax1, ax2) = plt.subplots(figsize=(18, 8), ncols=2)
# fig.suptitle("Potential Sites for Targeted Capacity Investments", fontsize=16,weight='bold')
# # Set axis off for both subplots
# ax1.set_axis_off()
# ax2.set_axis_off()

# # Shadow effect offset
# shadow_offset = 0.01

# # Plot solar map on ax1
# # Add shadow effect for solar map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax1, color='None', edgecolor='grey', linewidth=1, alpha=0.7)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# # Plot solar cells
# resource_clusters_solar.plot(column='solar_category', ax=ax1, cmap='Wistia', legend=True, 
#            legend_kwds={'title': "Solar Capacity (MW)", 'loc': 'upper right','fontsize':12,'bbox_to_anchor':(legend_x_ax_offset,1), 'frameon': False})

# # Plot actual boundary for solar map
# boundary.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)
# """
# # Annotate region numbers for solar map
# for idx, row in boundary.iterrows():
#     centroid = row.geometry.centroid
#     ax1.annotate(f"{row['Region_Number']}", 
#                  xy=(centroid.x, centroid.y), 
#                  ha='center', va='center',
#                  fontsize=7, color='black',
#                  bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
# """
# # Plot wind map on ax2
# # Add shadow effect for wind map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax2, color='None', edgecolor='grey', linewidth=1, alpha=0.7)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# # Plot wind cells
# resource_clusters_wind.plot(column='wind_category', ax=ax2, cmap='summer', legend=True, 
#            legend_kwds={'title': "Wind Capacity (MW)", 'fontsize':12,'bbox_to_anchor':(legend_x_ax_offset,1), 'frameon': False})

# # Plot actual boundary for wind map
# boundary.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.7)
# """
# # Annotate region numbers for wind map
# for idx, row in boundary.iterrows():
#     centroid = row.geometry.centroid
#     ax2.annotate(f"{row['Region_Number']}", 
#                  xy=(centroid.x, centroid.y), 
#                  ha='center', va='center',
#                  fontsize=8, color='black',
#                  bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
# """
# # Adjust layout for cleaner appearance
# fig.patch.set_alpha(0)  # Make figure background transparent
# plt.tight_layout()


# # Add annotation for solar capacity
# ax1.annotate(f"Targeted Capacity: \n{int(resource_clusters_solar.potential_capacity.sum()/1e3)} GW",
#              xy=(0.9, 0.6), xycoords='axes fraction', ha='center', 
#              fontsize=14, color='black', fontweight='bold')

# # Add annotation for wind capacity
# ax2.annotate(f"Targeted Capacity: \n{int(resource_clusters_wind.potential_capacity.sum()/1e3)} GW",
#              xy=(0.9, 0.6), xycoords='axes fraction', ha='center', 
#              fontsize=14, color='black', fontweight='bold')
# # Show the side-by-side plot

# plt.savefig('solar_wind_capacity_map.png',dpi=300)
# plt.show()

# Energy Calculations

### Proximity Range Filters

In [ ]:
# GRID_PROXIMITY_KM:int=30 #km
# cells_grid_proximity_filtered=cells[cells['nearest_station_distance_km']<=GRID_PROXIMITY_KM]

# solar_total_filtered=cells_grid_proximity_filtered['potential_capacity_solar'].sum()/1E3
# wind_total_filtered=cells_grid_proximity_filtered['potential_capacity_wind'].sum()/1E3

# print(f"Total potential solar capacity @ {GRID_PROXIMITY_KM}km grid proximity: {solar_total_filtered:.2f} GW")
# print(f"Total potential wind capacity @ {GRID_PROXIMITY_KM}km grid proximity: {wind_total_filtered:.2f} GW")

## Supply Curve Analytics

- Analysing 10000 GWh solar resource's supply curve

In [ ]:
for policy in POLICYs:
    
    cells=BC_dfs_all_runs[f'{policy}']['cells']
    
    print(f"Policy: {policy}")
    print("--" * 50)
    cells['potential_energy_solar_GWh']=cells['potential_capacity_solar']*cells['solar_CF_mean']*8760/1E3 # GWh
    cells['potential_energy_wind_GWh']=cells['potential_capacity_wind']*cells['wind_CF_mean']*8760/1E3 # GWh

    wind_total=cells['potential_capacity_wind'].sum()/1E3
    solar_total=cells['potential_capacity_solar'].sum()/1E3

    print(f"Total potential solar capacity: {solar_total:.2f} GW")
    print(f"Total potential wind capacity: {wind_total:.2f} GW\n")
    print("." * 10)
    
    cells_solar_sorted = cells.sort_values(by='lcoe_solar')
    solar_cumsum = cells_solar_sorted['potential_energy_solar_GWh'].cumsum()
    print("<=10,000 GWh from Solar")
    print("." * 10)
    cells_until_10000_solar = cells_solar_sorted.loc[solar_cumsum <= 10000]

    print(f"Total potential solar energy: {cells_until_10000_solar['potential_energy_solar_GWh'].sum():.2f} GWh")
    print(f"Number of cells needed (solar): {len(cells_until_10000_solar)}")
    print(f"{cells_until_10000_solar.potential_capacity_solar.sum()/1E3:.2f} GW potential solar capacity")
    print(f"Mean Score : {cells_until_10000_solar.lcoe_solar.mean():.2f} $/MWH")
    cells_until_10000_solar.lcoe_solar.describe()
    print("." * 10)
    cells_wind_sorted = cells.sort_values(by='lcoe_wind')
    wind_cumsum = cells_wind_sorted['potential_energy_wind_GWh'].cumsum()
    cells_until_40000_wind = cells_wind_sorted.loc[wind_cumsum <= 40000]
    print("<=40,000 GWh from Wind")
    print("." * 10)
    print(f"Total potential wind energy: {cells_until_40000_wind['potential_energy_wind_GWh'].sum():.2f} GWh")
    
    print(f"Number of cells needed (wind): {len(cells_until_40000_wind)}")
    print(f" {cells_until_40000_wind.potential_capacity_wind.sum()/1E3:.2f} GW potential wind capacity")
    print(f" Mean Score : {cells_until_40000_wind.lcoe_wind.mean():.2f} $/MWH")
    cells_until_40000_wind.lcoe_wind.describe()
    print('\n')
    
    
# cells_solar_sorted = cells.sort_values(by='lcoe_solar')
# solar_cumsum = cells_solar_sorted['potential_energy_solar_GWh'].cumsum()
# cells_until_10000_solar = cells_solar_sorted.loc[solar_cumsum <= 10000]

# print(f"Total potential solar energy: {cells_until_10000_solar['potential_energy_solar_GWh'].sum():.2f} GWh")
# print(f"Number of cells needed (solar): {len(cells_until_10000_solar)}")
# print(f" {cells_until_10000_solar.potential_capacity_solar.sum()/1E3:.2f} GW potential solar capacity")
# print(f" Mean Score : {cells_until_10000_solar.lcoe_solar.mean():.2f} $/MWH")
# cells_until_10000_solar.lcoe_solar.describe()

- Analysing 40000 GWh wind resource's supply curve

In [ ]:
lcoe_threshold_solar:int=57 #$/MWH
lcoe_threshold_wind:int=50 #$/MWH
for policy in POLICYs:
    print(f"Policy: {policy}")
    print("--" * 50)
    cells=BC_dfs_all_runs[f'{policy}']['cells']
    
    cells['potential_energy_solar_GWh']=cells['potential_capacity_solar']*cells['solar_CF_mean']*8760/1E3 # GWh
    cells['potential_energy_wind_GWh']=cells['potential_capacity_wind']*cells['wind_CF_mean']*8760/1E3 # GWh

    wind_total=cells['potential_capacity_wind'].sum()/1E3
    solar_total=cells['potential_capacity_solar'].sum()/1E3

    print(f"Total potential solar capacity: {solar_total:.2f} GW")
    print(f"Total potential wind capacity: {wind_total:.2f} GW\n")
    print("." * 10)
    
    solar_cells_filtered = cells[cells['lcoe_solar'] <= lcoe_threshold_solar]
    wind_cells_filtered = cells[cells['lcoe_wind'] <= lcoe_threshold_wind]
    
    print(f"Total potential solar capacity <={lcoe_threshold_solar} $/MWh LCOE threshold: {solar_cells_filtered['potential_capacity_solar'].sum()/1E3:.2f} GW")
    print(f"Total potential wind capacity <={lcoe_threshold_wind} $/MWh LCOE threshold: { wind_cells_filtered['potential_capacity_wind'].sum()/1E3:.2f} GW")


    print(f"Total potential solar energy @ <={lcoe_threshold_solar} $/MWh LCOE threshold: {solar_cells_filtered['potential_energy_solar_GWh'].sum():.2f} GWh")
    print(f"Total potential wind energy @ <={lcoe_threshold_wind} $/MWh LCOE threshold: {wind_cells_filtered['potential_energy_wind_GWh'].sum():.2f} GWh")
    print('\n')
    
    if policy=='BASELINE':
        solar_cells_filtered.to_csv(f"{BASELINE_results_save_to}/solar_cells_below_{lcoe_threshold_solar}_$pMWh_{region_code}_{policy}.csv",index=False)
        wind_cells_filtered.to_csv(f"{BASELINE_results_save_to}/wind_cells_below_{lcoe_threshold_wind}_$pMWh_{region_code}_{policy}.csv",index=False)
    else:
        solar_cells_filtered.to_csv(f"{POLICY_results_save_to}/solar_cells_below_{lcoe_threshold_solar}_$pMWh_{region_code}_{POLICY}.csv",index=False)
        wind_cells_filtered.to_csv(f"{POLICY_results_save_to}/wind_cells_below_{lcoe_threshold_wind}_$pMWh_{region_code}_{POLICY}.csv",index=False)


- Load BASELINE cells

In [ ]:
solar_cells_filtered_default= pd.read_csv(f"{BASELINE_results_save_to}/solar_cells_below_{lcoe_threshold_solar}_$pMWh_BC_BASELINE.csv")
wind_cells_filtered_default = pd.read_csv(f"{BASELINE_results_save_to}/wind_cells_below_{lcoe_threshold_wind}_$pMWh_BC_BASELINE.csv")

- Compare with Default (baseline)

In [ ]:
import matplotlib.pyplot as plt

# --- Prepare supply curve data (technical potential) ---
solar_sorted = solar_cells_filtered.sort_values(by='lcoe_solar')
solar_cumsum_energy = solar_sorted['potential_energy_solar_GWh'].cumsum()
solar_lcoe = solar_sorted['lcoe_solar']

wind_sorted = wind_cells_filtered.sort_values(by='lcoe_wind')
wind_cumsum_energy = wind_sorted['potential_energy_wind_GWh'].cumsum()
wind_lcoe = wind_sorted['lcoe_wind']

# --- Prepare policy-constrained curves ---
solar_sorted_default = solar_cells_filtered_default.sort_values(by='lcoe_solar')
solar_cumsum_energy_default = solar_sorted_default['potential_energy_solar_GWh'].cumsum()
solar_lcoe_default = solar_sorted_default['lcoe_solar']

wind_sorted_default = wind_cells_filtered_default.sort_values(by='lcoe_wind')
wind_cumsum_energy_default = wind_sorted_default['potential_energy_wind_GWh'].cumsum()
wind_lcoe_default = wind_sorted_default['lcoe_wind']

# --- Plot ---

fig, ax = plt.subplots(figsize=(9, 5), dpi=500)

# Policy-constrained (emphasized solid lines)
ax.plot(
    solar_cumsum_energy, solar_lcoe,
    label='Solar (Policy-constrained)',
    color="#DA391D", linestyle='-', linewidth=2.2,alpha=0.8
)
ax.plot(
    wind_cumsum_energy, wind_lcoe,
    label='Wind (Policy-constrained)',
    color="#530c9a", linestyle='-', linewidth=2.2,alpha=0.8
)

# Baseline (lighter dashed lines)
ax.plot(
    solar_cumsum_energy_default, solar_lcoe_default,
    label='Solar (Baseline)',
    color="#F494346D", linestyle='--', linewidth=3.0, alpha=1,
    zorder=2
)
ax.plot(
    wind_cumsum_energy_default, wind_lcoe_default,
    label='Wind (Baseline)',
    color="#b165fc3c", linestyle='--', linewidth=3.0, alpha=1,
    zorder=2
)

# Labels and title
ax.set_xlabel('Cumulative Potential Energy (GWh)', fontsize=12, labelpad=8)
ax.set_ylabel('Score [Normalized LCOE ($/MWh)]', fontsize=12, labelpad=8)
ax.set_title(
    'Supply Curves for Solar and Wind Resources\nBaseline vs. Policy-Constrained Potential',
    fontsize=14, weight='bold', pad=12
)

# Optional axis limits (if you want to emphasize a range)
# ax.set_xlim(0, 20000)

# Legend
ax.legend(
    frameon=False,
    fontsize=12,
    loc='upper right',
    bbox_to_anchor=(0.98, 0.45),  # slightly lower for better spacing
    # title="Scenario",
    title_fontsize=11
)

# Clean spines
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)

# Gridlines for readability
ax.grid(True, which='both', linestyle=':', linewidth=0.7, color='gray', alpha=0.4)
# Set x-axis limit to 40,000 GWh
ax.set_xlim(0, 40000)
# Make tick mark font bigger
ax.tick_params(axis='both', which='major', labelsize=13)
# Tight layout
plt.tight_layout()
# plt.show()
plt.savefig(f"../vis/{country_kwd}/{region_code}/supply_curve_baseline_vs_policy_{region_code}.svg", bbox_inches='tight', transparent=False)

plt.savefig(f"../docs/source/_static/supply_curve_baseline_vs_policy_{region_code}.png", bbox_inches="tight")

- Extract the co-located Resources (cells)

In [ ]:
common_idx = solar_cells_filtered.index.intersection(wind_cells_filtered.index)
solar_common_capacity = solar_cells_filtered.loc[common_idx, 'potential_capacity_solar'].sum() / 1E3
wind_common_capacity = wind_cells_filtered.loc[common_idx, 'potential_capacity_wind'].sum() / 1E3
timeseries_solar_filtered=BC_dfs_all_runs[f'{POLICY}']['timeseries_solar'][common_idx]
timeseries_wind_filtered=BC_dfs_all_runs[f'{POLICY}']['timeseries_wind'][common_idx]
print(f"Common index count: {len(common_idx)}")
print(f"Solar capacity (GW) in common cells: {solar_common_capacity:.2f}")
print(f"Wind capacity (GW) in common cells: {wind_common_capacity:.2f}")

# Temporal Analysis

## Complimentarity

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle


# -----------------------------------------
# 1) DATA: compute both matrices + scores
# -----------------------------------------
def compute_region_complementarity(
    timeseries_solar: pd.DataFrame,
    timeseries_wind:  pd.DataFrame,
    region: str | None = None,
    extract_region_fn=None,  # callable: str -> region label; optional
):
    """
    Returns both hour×month matrices + complementarity scores.

    Outputs dict:
      {
        'meta': {'region_label', 'n_sites'},
        'pearson': {'matrix': pearson_mat (24×12), 'pearson_C': float},
        'diff':    {'matrix': diff_mat (24×12),    'diff_C': float}
      }
    """

    # --- Align indices ---
    idx = timeseries_solar.index.intersection(timeseries_wind.index)
    if len(idx) == 0:
        raise ValueError("Solar and wind time indices do not overlap.")
    ts_solar_full = timeseries_solar.loc[idx]
    ts_wind_full  = timeseries_wind.loc[idx]

    # --- Region filter helper ---
    def _select_cols(cols, region_label):
        if region_label is None:
            return list(cols)
        if extract_region_fn is not None:
            return [c for c in cols if extract_region_fn(c) == region_label]
        # fallback: substring match
        region_label = str(region_label)
        return [c for c in cols if region_label in str(c)]

    solar_cols = _select_cols(ts_solar_full.columns, region)
    wind_cols  = _select_cols(ts_wind_full.columns,  region)
    common_cols = sorted(set(solar_cols).intersection(set(wind_cols)))
    if len(common_cols) == 0:
        raise ValueError(f"No common site columns for region={region!r}. Check column names / extract_region().")

    # --- Aggregate across selected sites ---
    solar_mean = ts_solar_full[common_cols].mean(axis=1)
    wind_mean  = ts_wind_full[common_cols].mean(axis=1)

    # --- Helper: build hour×month matrix(s) ---
    def _to_hour_month_matrix(series: pd.Series) -> pd.DataFrame:
        df = pd.DataFrame({'value': series})
        df['hour'] = df.index.hour
        df['month'] = df.index.month
        mat = df.groupby(['hour','month'], observed=True)['value'].mean().unstack('month')
        return mat.reindex(index=range(24), columns=range(1, 13))

    def _hour_month_corr(a: pd.Series, b: pd.Series) -> pd.DataFrame:
        df = pd.DataFrame({'solar': a, 'wind': b}).dropna()
        df['hour'] = df.index.hour
        df['month'] = df.index.month

        def _corr(g):
            if (g['solar'].std(ddof=0) == 0) or (g['wind'].std(ddof=0) == 0):
                return np.nan
            return g['solar'].corr(g['wind'])

        r = df.groupby(['hour', 'month'], observed=True).apply(_corr).unstack('month')
        return r.reindex(index=range(24), columns=range(1, 13))

    # --- Matrices ---
    pearson_mat = _hour_month_corr(solar_mean, wind_mean)                 # in [-1, 1]
    solar_mat   = _to_hour_month_matrix(solar_mean)
    wind_mat    = _to_hour_month_matrix(wind_mean)

    # Normalize for dominance map
    solar_norm = solar_mat / np.nanmax(solar_mat.values)
    wind_norm  = wind_mat  / np.nanmax(wind_mat.values)
    diff_mat   = solar_norm - wind_norm                                   # [-1, 1] approx

    # --- Complementarity scores (scalars) ---
    # Pearson-based complementarity: average anti-correlation (negative r)
    pearson_C = np.nanmean(np.clip(-pearson_mat.values, 0, 1))            # 0–1 (higher = more complementary)

    # Diff-based complementarity: average absolute dominance difference
    diff_C = np.nanmean(np.abs(diff_mat.values))                          # 0–1 (higher = stronger diurnal-seasonal offset)

    return {
        'meta': {
            'region_label': region if region is not None else "All Regions",
            'n_sites': len(common_cols),
        },
        'pearson': {
            'matrix': pearson_mat,
            'pearson_C': float(pearson_C),
        },
        'diff': {
            'matrix': diff_mat,
            'diff_C': float(diff_C),
        }
    }

# -----------------------------------------
# 2) PLOT: consume any precomputed matrix
# -----------------------------------------
def plot_complementarity_heatmap(
    mat: pd.DataFrame,
    metric: str = "pearson",                   # "pearson" or "diff"
    region_label: str = "All Regions",
    n_sites: int | None = None,
    clusters: bool = True,
    font_family: str | None = None,
    style_path: str | None = None,
    show: bool = True,
):
    sns.set_theme(style="whitegrid")
    sns.set_palette("Set2")
    if font_family is not None:
        plt.rcParams['font.family'] = font_family
    if style_path:
        try:
            plt.style.use(style_path)
        except Exception:
            pass

    # Diverging cmap for "diff" metric (as in your earlier code)
    colors_neg = plt.cm.PuBu(np.linspace(0.3, 1, 128))
    colors_pos = plt.cm.OrRd(np.linspace(0.3, 1, 128))
    colors = np.vstack((colors_neg[::-1], colors_pos))
    custom_cmap = LinearSegmentedColormap.from_list('WindSolarDiv', colors)

    # Ranges and labels
    if metric.lower() == "pearson":
        vmin, vmax, center = -1.0, 1.0, 0.0
        cmap = "coolwarm"
        cbar_label = "Pearson r (negative = complementary)"
        title_metric = "Pearson r"
    elif metric.lower() == "diff":
        vmin, vmax, center = -0.8, 0.8, 0.0
        cmap = custom_cmap
        cbar_label = 'Resource Dominance\n(Purple: Wind | Orange: Solar)'
        title_metric = "Solar–Wind Dominance"
    else:
        raise ValueError("metric must be 'pearson' or 'diff'.")

    month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
    fig, ax = plt.subplots(figsize=(9, 5), dpi=500)
    sns.heatmap(
        mat, cmap=cmap, vmin=vmin, vmax=vmax, center=center,
        linewidths=0, cbar_kws={'label': cbar_label, 'shrink': 0.8}, ax=ax
    )

    ax.set_xticks(np.arange(12) + 0.5)
    ax.set_xticklabels(month_names, rotation=90, fontsize=9)
    ax.set_yticks(np.arange(0, 24, 4) + 0.5)
    ax.set_yticklabels(np.arange(0, 24, 4), fontsize=9)
    ax.set_xlabel('Month', fontsize=9, fontweight='bold')
    ax.set_ylabel('Hour of Day', fontsize=9, fontweight='bold')
    ax.set_title(f'{region_label} | {title_metric}', fontsize=14, fontweight='bold')

    # Annotation badge
    if n_sites is not None:
        annotation_custom = 'Representative Clustered Sites Nos:' if clusters else 'ERA5 Cells Nos:'
        ax.text(0.02, 0.97, f'{annotation_custom} {n_sites}', transform=ax.transAxes,
                ha='left', va='bottom', fontsize=8, fontweight='bold', color='black',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='lightgrey', alpha=0.7))

    # KEEP OLD ANNOTATIONS for "diff"
    if metric.lower() == "diff":
        # Peak solar and wind periods (original rectangles & labels)
        ax.add_patch(Rectangle((0,10),12,6, linewidth=1.5, edgecolor='orange',
                               facecolor='none', linestyle='--', alpha=0.8))
        ax.text(6, 8.5, 'Solar Peak\nDay', ha='center', va='center', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='orange', alpha=0.7))

        ax.add_patch(Rectangle((0,0),12,6, linewidth=1.5, edgecolor='lightblue',
                               facecolor='none', linestyle='--', alpha=0.8))
        ax.text(6, 3, 'Wind Dominant\nNight', ha='center', va='center', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='lightblue', alpha=0.7))

        ax.add_patch(Rectangle((0,18),12,6, linewidth=1.5, edgecolor='lightblue',
                               facecolor='none', linestyle='--', alpha=0.8))
        ax.text(6, 21, 'Wind Dominant\nEve', ha='center', va='center', fontsize=7, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='lightblue', alpha=0.7))

        # Seasonal annotations (kept as in prior code)
        ax.text(1.5, 28.2, 'Winter', ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightcyan', alpha=0.8))
        ax.text(5.5, 28.2, 'Summer', ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))
        ax.text(9.5, 28.2, 'Fall', ha='center', va='center', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgreen', alpha=0.8))

    plt.tight_layout()
    if show:
        plt.show()
    return fig, ax

# -----------------------------------------
# 3) Optional wrapper with old signature
# -----------------------------------------
def plot_region_complementarity(
    timeseries_solar: pd.DataFrame,
    timeseries_wind:  pd.DataFrame,
    region_code: str,
    RUN_ID: str = 'default',
    region: str | None = None,
    clusters: bool = True,
    aggregate: bool = False,   # kept for signature compatibility
    show: bool = True,
    font_family: str | None = None,
    metric: str = "pearson",
    extract_region_fn=None,
    style_path: str | None = None,
):
    results = compute_region_complementarity(
        timeseries_solar, timeseries_wind, region=region, extract_region_fn=extract_region_fn
    )
    if metric.lower() == "pearson":
        mat = results['pearson']['matrix']
        cval = results['pearson']['pearson_C']
        title = f"{results['meta']['region_label']} | Pearson r | Complementarity Score: {cval:.2f}"
    elif metric.lower() == "diff":
        mat = results['diff']['matrix']
        cval = results['diff']['diff_C']
        title = f"{results['meta']['region_label']} | Solar–Wind Dominance | Complementarity Score: {cval:.2f}"
    else:
        raise ValueError("metric must be 'pearson' or 'diff'.")

    fig, ax = plot_complementarity_heatmap(
        mat=mat, metric=metric, region_label=results['meta']['region_label'],
        n_sites=results['meta']['n_sites'], clusters=clusters,
        font_family=font_family, style_path=style_path, show=show
    )
    ax.set_title(title, fontsize=14, fontweight='bold')
    return results, (fig, ax)


In [ ]:
timeseries_solar=BC_dfs_all_runs[f'{POLICY}']['timeseries_solar']
timeseries_wind=BC_dfs_all_runs[f'{POLICY}']['timeseries_wind']

In [ ]:
results, _ = plot_region_complementarity(
    timeseries_solar, timeseries_wind,
    region_code="BC", RUN_ID="v1",
    region="PeaceRiver",
    metric="diff", show=False,
    extract_region_fn=vis.extract_region
)


pearson_C = results['pearson']['pearson_C']
diff_C    = results['diff']['diff_C']


In [ ]:
from typing import Literal, Optional

import numpy as np
import pandas as pd


# ---------- helpers ----------
def _to_hour_month_matrix(series: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame({'value': series})
    df['hour'] = df.index.hour
    df['month'] = df.index.month
    mat = df.groupby(['hour','month'], observed=True)['value'].mean().unstack('month')
    return mat.reindex(index=range(24), columns=range(1, 13))

def _hour_month_corr(a: pd.Series, b: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame({'solar': a, 'wind': b}).dropna()
    df['hour'] = df.index.hour
    df['month'] = df.index.month

    def _corr(g):
        if (g['solar'].std(ddof=0) == 0) or (g['wind'].std(ddof=0) == 0):
            return np.nan
        return g['solar'].corr(g['wind'])

    r = df.groupby(['hour','month'], observed=True).apply(_corr).unstack('month')
    return r.reindex(index=range(24), columns=range(1, 13))

def _build_weights(
    weighting: Literal["none","demand","cf","energy"] = "none",
    demand_ts: Optional[pd.Series] = None,
    solar_mat: Optional[pd.DataFrame] = None,
    wind_mat: Optional[pd.DataFrame] = None
) -> pd.DataFrame:
    """
    Returns non-negative 24x12 weights that sum to 1 (NaNs treated as 0).
    - "none": uniform weights
    - "demand": average demand in each (hour,month) bin
    - "cf":     weight by average of (solar_mat + wind_mat) normalized by their own max (capacity-factor-like)
    - "energy": weight by absolute energy contribution proxy (solar_mat + wind_mat) without normalization
    """
    W = pd.DataFrame(1.0, index=range(24), columns=range(1,13))
    if weighting == "none":
        pass
    elif weighting == "demand":
        if demand_ts is None:
            raise ValueError("Provide demand_ts for weighting='demand'.")
        dd = pd.DataFrame({'d': demand_ts})
        dd['hour']  = dd.index.hour
        dd['month'] = dd.index.month
        W = dd.groupby(['hour','month'], observed=True)['d'].mean().unstack('month')
    elif weighting in ("cf","energy"):
        if solar_mat is None or wind_mat is None:
            raise ValueError("Provide solar_mat and wind_mat for weighting in {'cf','energy'}.")
        if weighting == "cf":
            denom_s = np.nanmax(solar_mat.values) or 1.0
            denom_w = np.nanmax(wind_mat.values) or 1.0
            s_norm  = solar_mat / denom_s
            w_norm  = wind_mat  / denom_w
            W = (s_norm + w_norm) / 2.0
        else:  # "energy"
            W = (solar_mat.fillna(0) + wind_mat.fillna(0))
    else:
        raise ValueError("weighting must be one of {'none','demand','cf','energy'}")

    W = W.fillna(0).clip(lower=0)
    s = W.to_numpy().sum()
    if s <= 0:
        # fallback to uniform if weights degenerate
        W.loc[:,:] = 1.0 / W.size
    else:
        W = W / s
    return W

def _dominance_matrix(
    solar_mat: pd.DataFrame,
    wind_mat:  pd.DataFrame,
    norm: Literal["max","p95"] = "max"
) -> pd.DataFrame:
    """Build solar–wind dominance matrix with robust normalization."""
    if norm == "max":
        denom_s = np.nanmax(solar_mat.values) or 1.0
        denom_w = np.nanmax(wind_mat.values) or 1.0
    elif norm == "p95":
        denom_s = np.nanpercentile(solar_mat.values, 95) or 1.0
        denom_w = np.nanpercentile(wind_mat.values, 95) or 1.0
    else:
        raise ValueError("norm must be 'max' or 'p95'")
    s_norm = solar_mat / denom_s
    w_norm = wind_mat  / denom_w
    return s_norm - w_norm

def _weighted_mean_abs(mat: pd.DataFrame, W: pd.DataFrame) -> float:
    A = np.abs(mat.values)
    w = W.values
    return float(np.nansum(A * w))

def _weighted_mean_neg_r(pearson_mat: pd.DataFrame, W: pd.DataFrame) -> float:
    X = np.clip(-pearson_mat.values, 0, 1)
    w = W.values
    return float(np.nansum(np.where(np.isnan(X), 0, X) * w))

# ---------- main robust computation ----------
def compute_complementarity_robust(
    ts_solar: pd.DataFrame,
    ts_wind:  pd.DataFrame,
    region: Optional[str] = None,
    extract_region_fn=None,
    weighting: Literal["none","demand","cf","energy"] = "none",
    demand_ts: Optional[pd.Series] = None,
    norm: Literal["max","p95"] = "p95",
):
    """
    Returns matrices and weighted scores using robust options.
    """
    # align
    idx = ts_solar.index.intersection(ts_wind.index)
    if len(idx) == 0:
        raise ValueError("Solar and wind time indices do not overlap.")
    ts_solar = ts_solar.loc[idx]
    ts_wind  = ts_wind.loc[idx]

    # region columns
    def _sel(cols, r):
        if r is None: return list(cols)
        if extract_region_fn is not None:
            return [c for c in cols if extract_region_fn(c) == r]
        r = str(r)
        return [c for c in cols if r in str(c)]

    sc = _sel(ts_solar.columns, region)
    wc = _sel(ts_wind.columns,  region)
    cols = sorted(set(sc).intersection(wc))
    if not cols:
        raise ValueError(f"No common site columns for region={region!r}.")

    # aggregate across selected sites
    s_mean = ts_solar[cols].mean(axis=1)
    w_mean = ts_wind[cols].mean(axis=1)

    pearson_mat = _hour_month_corr(s_mean, w_mean)
    solar_mat   = _to_hour_month_matrix(s_mean)
    wind_mat    = _to_hour_month_matrix(w_mean)
    diff_mat    = _dominance_matrix(solar_mat, wind_mat, norm=norm)

    # weights
    W = _build_weights(weighting=weighting, demand_ts=demand_ts,
                       solar_mat=solar_mat, wind_mat=wind_mat)

    pearson_C = _weighted_mean_neg_r(pearson_mat, W)   # 0–1
    diff_C    = _weighted_mean_abs(diff_mat,    W)     # 0–1-ish

    return {
        'meta': {'region_label': region or "All Regions", 'n_sites': len(cols)},
        'pearson': {'matrix': pearson_mat, 'pearson_C': pearson_C},
        'diff':    {'matrix': diff_mat,    'diff_C': diff_C},
        'weights': W,
        'norm': norm,
        'weighting': weighting
    }

# ---------- bootstrap over sites ----------
def bootstrap_complementarity(
    ts_solar: pd.DataFrame,
    ts_wind:  pd.DataFrame,
    region: Optional[str] = None,
    extract_region_fn=None,
    weighting: Literal["none","demand","cf","energy"] = "none",
    demand_ts: Optional[pd.Series] = None,
    norm: Literal["max","p95"] = "p95",
    B: int = 500,
    random_state: Optional[int] = 0
):
    """
    Resamples site columns with replacement and recomputes weighted scores.
    Returns point estimates + 95% CIs and the bootstrap series.
    """
    rng = np.random.default_rng(random_state)

    # restrict to region once to define pool
    if region is None:
        pool = sorted(set(ts_solar.columns).intersection(ts_wind.columns))
    else:
        def _match(cols):
            if extract_region_fn is not None:
                return [c for c in cols if extract_region_fn(c) == region]
            return [c for c in cols if str(region) in str(c)]
        pool = sorted(set(_match(ts_solar.columns)).intersection(_match(ts_wind.columns)))

    if not pool:
        raise ValueError(f"No common site columns for region={region!r}.")

    idx = ts_solar.index.intersection(ts_wind.index)
    ts_solar = ts_solar.loc[idx, pool]
    ts_wind  = ts_wind.loc[idx,  pool]

    def _one_estimate(cols):
        s_mean = ts_solar[cols].mean(axis=1)
        w_mean = ts_wind[cols].mean(axis=1)
        pearson_mat = _hour_month_corr(s_mean, w_mean)
        solar_mat   = _to_hour_month_matrix(s_mean)
        wind_mat    = _to_hour_month_matrix(w_mean)
        diff_mat    = _dominance_matrix(solar_mat, wind_mat, norm=norm)
        W = _build_weights(weighting=weighting, demand_ts=demand_ts,
                           solar_mat=solar_mat, wind_mat=wind_mat)
        return (
            _weighted_mean_neg_r(pearson_mat, W),
            _weighted_mean_abs(diff_mat,    W)
        )

    # point estimate with all unique cols
    pe_pearson, pe_diff = _one_estimate(pool)

    # bootstrap
    pears, diffs = [], []
    n = len(pool)
    for _ in range(B):
        sample_cols = list(rng.choice(pool, size=n, replace=True))
        p, d = _one_estimate(sample_cols)
        pears.append(p); diffs.append(d)

    pears = np.array(pears); diffs = np.array(diffs)
    ci_p = (np.nanpercentile(pears, 2.5), np.nanpercentile(pears, 97.5))
    ci_d = (np.nanpercentile(diffs, 2.5), np.nanpercentile(diffs, 97.5))

    return {
        'point': {'pearson_C': pe_pearson, 'diff_C': pe_diff},
        'ci95':  {'pearson_C': ci_p, 'diff_C': ci_d},
        'boot':  {'pearson_C': pears, 'diff_C': diffs},
        'meta':  {'region_label': region or "All Regions", 'n_sites': n,
                  'B': B, 'norm': norm, 'weighting': weighting}
    }


In [ ]:
boundary = boundary.copy()
boundary['Region'] = boundary['Region'].str.replace(" ", "")
len(boundary)

In [ ]:
import geopandas as gpd

# Precompute centroids to avoid repeated calculation and warnings
boundary_centroids = boundary.geometry.centroid
rob = {}

for idx, (region, centroid) in enumerate(zip(boundary['Region'], boundary_centroids)):
    rob[region] = compute_complementarity_robust(
        ts_solar=timeseries_solar,
        ts_wind=timeseries_wind,
        region=region,
        extract_region_fn=vis.extract_region,
        weighting="cf",           # "none", "demand", "cf", or "energy"
        # demand_ts=demand_series,      # pd.Series indexed like the timeseries
        norm="p95"                    # robust dominance normalization
    )
    # You can use centroid.x, centroid.y if needed for plotting or annotation


In [ ]:
scores = pd.DataFrame([
    {
        'region_code': region_code,
        'Region': region,
        'run_id': POLICY,
        'n_sites': result['meta']['n_sites'],
        'pearson_C': result['pearson']['pearson_C'],
        'diff_C': result['diff']['diff_C'],
        'weighting': result['weighting'],
        'norm': result['norm'],
    }
    for region, result in rob.items()
])
scores

In [ ]:
gdf_out = boundary.merge(scores, on="Region", how="inner")
gdf_out = gdf_out.merge(cells_aggrs_region_scenario, on='Region',how="inner")

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Patch, Wedge


def make_pie_image(solar_share, size=0.2):
    """Return a matplotlib image array of a pie chart (solar vs wind)."""
    fig, ax = plt.subplots(figsize=(1,1), dpi=500)
    ax.pie([solar_share, 1-solar_share],
           colors=["orange", "royalblue"],
           startangle=90,
           counterclock=False,
           wedgeprops=dict(edgecolor="white", linewidth=0.4))
    ax.set_aspect("equal")
    plt.axis("off")

    buf = BytesIO()
    plt.savefig(buf, format="png", bbox_inches="tight", transparent=True)
    plt.close(fig)
    buf.seek(0)
    return plt.imread(buf)

def plot_pie_and_score_inset(gdf, size_scale=5, cmap="Blues"):
    gdf = gdf.copy()
    gdf['total_capacity'] = gdf['potential_capacity_solar'] + gdf['potential_capacity_wind']
    gdf['solar_share'] = gdf['potential_capacity_solar'] / gdf['total_capacity']

    centroids = gdf.geometry.centroid

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,6), dpi=500)

    # --- Left: Pie inset bubbles ---
    gdf.boundary.plot(ax=ax1, color="black", linewidth=0.5)
    for _, row in gdf.iterrows():
        if row['total_capacity'] <= 0: continue
        cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        solar_share = row['solar_share']

        pie_img = make_pie_image(solar_share)
        zoom = np.sqrt(row['total_capacity']) * size_scale / 1000.0

        ab = AnnotationBbox(OffsetImage(pie_img, zoom=zoom),
                            (cx, cy), frameon=False)
        ax1.add_artist(ab)

    ax1.set_title("Solar/Wind Mix (Pie, Bubble Size = Capacity)")
    ax1.set_axis_off()

    # Add legend for pie colors
    pie_handles = [Patch(facecolor="orange", edgecolor="k", label="Solar"),
                   Patch(facecolor="royalblue", edgecolor="k", label="Wind")]
    ax1.legend(handles=pie_handles, loc="lower left", frameon=False, title="Resource")

    # --- Right: Bubble map with color ---
    gdf.boundary.plot(ax=ax2, color="black", linewidth=0.5)
    sc = ax2.scatter(centroids.x, centroids.y,
                     s=np.sqrt(gdf['total_capacity'])*(size_scale+1.2),
                     c=gdf['complementarity_score'], cmap=cmap,
                     edgecolor="k", linewidth=0.4, alpha=0.9)
    cbar = plt.colorbar(sc, ax=ax2, shrink=0.7, pad=0.02)
    cbar.set_label("Complementarity Score")
    ax2.set_title("Complementarity Score (Bubble Size = Capacity)")
    ax2.set_axis_off()

    # --- Shared bubble size legend ---
    for ax in [ax1, ax2]:
        for s in [1000, 5000, 10000]:  # example MW thresholds
            ax.scatter([], [], s=np.sqrt(s) * size_scale,
                       facecolors='none', edgecolors='k', label=f"{s} MW")
    ax2.legend(scatterpoints=1, frameon=False, labelspacing=1,
               loc='lower left', title="Total Capacity")

    plt.tight_layout()
    return fig


In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Patch


def make_pie_image(solar_share, border_color, size=0.2, border_width=8):
    """Return a matplotlib image array of a pie chart with colored border."""
    fig, ax = plt.subplots(figsize=(1,1), dpi=500)
    
    # Create pie chart
    wedges, texts = ax.pie([solar_share, 1-solar_share],
                          colors=["orange", "royalblue"],
                          startangle=90,
                          counterclock=False,
                          wedgeprops=dict(edgecolor=border_color, 
                                        linewidth=border_width))
    
    ax.set_aspect("equal")
    plt.axis("off")
    
    # Save to buffer
    buf = BytesIO()
    plt.savefig(buf, format="svg", bbox_inches="tight", transparent=True)
    plt.close(fig)
    buf.seek(0)
    return plt.imread(buf)

def plot_combined_pie_and_score(gdf, size_scale=5, cmap="Blues"):
    """Combined plot with pie charts sized by capacity and bordered by complementarity score."""
    gdf = gdf.copy()
    gdf['total_capacity'] = gdf['potential_capacity_solar'] + gdf['potential_capacity_wind']
    gdf['solar_share'] = gdf['potential_capacity_solar'] / gdf['total_capacity']
    
    # Normalize complementarity scores for color mapping
    norm = Normalize(vmin=gdf['complementarity_score'].min(), 
                    vmax=gdf['complementarity_score'].max())
    cmap_obj = plt.get_cmap(cmap)
    
    centroids = gdf.geometry.centroid
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8), dpi=500)
    
    # Plot boundaries
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.5)
    
    # Add pie charts with score-colored borders
    for _, row in gdf.iterrows():
        if row['total_capacity'] <= 0: 
            continue
            
        cx, cy = row.geometry.centroid.x, row.geometry.centroid.y
        solar_share = row['solar_share']
        score = row['complementarity_score']
        
        # Get border color from complementarity score
        border_color = cmap_obj(norm(score))
        
        # Create pie image
        pie_img = make_pie_image(solar_share, border_color)
        zoom = np.sqrt(row['total_capacity']) * size_scale / 1000.0
        
        # Add to plot
        ab = AnnotationBbox(OffsetImage(pie_img, zoom=zoom),
                           (cx, cy), frameon=False)
        ax.add_artist(ab)
    
    ax.set_title("Solar/Wind Mix with Complementarity Score Borders\n(Pie Size = Total Capacity)")
    ax.set_axis_off()
    
    # Create legends
    # 1. Pie chart legend (solar/wind)
    pie_handles = [Patch(facecolor="orange", edgecolor="k", label="Solar"),
                   Patch(facecolor="royalblue", edgecolor="k", label="Wind")]
    legend1 = ax.legend(handles=pie_handles, loc="upper left", frameon=False, 
                       title="Energy Source")
    
    # 2. Size legend (capacity)
    size_handles = []
    for capacity in [1000, 5000, 10000]:  # example MW thresholds
        size_handles.append(plt.Line2D([0], [0], marker='o', color='w', 
                                     markerfacecolor='gray', markersize=np.sqrt(capacity)*size_scale/200,
                                     markeredgecolor='k', label=f"{capacity} MW"))
    
    legend2 = ax.legend(handles=size_handles, loc="lower left", frameon=False,
                       title="Total Capacity")
    
    # Add both legends to plot
    ax.add_artist(legend1)
    
    # 3. Colorbar for complementarity score
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.6, pad=0.02)
    cbar.set_label("Complementarity Score (Border Color)")
    
    plt.tight_layout()
    return fig

In [ ]:
# timeseries_solar_filtered=timeseries_solar[common_idx]
# timeseries_wind_filtered=timeseries_wind[common_idx]
# vis.plot_region_complementarity(timeseries_solar=timeseries_solar, 
#                                 timeseries_wind=timeseries_wind,
#                                 region_code=region_code,
#                                 RUN_ID=RUN_ID,
#                                 region='PeaceRiver',
#                                 # clusters=True,
#                                 metric='pearson',
#                                 aggregate=True
#                                 )

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.cm import ScalarMappable, get_cmap
from matplotlib.colors import Normalize

plt.rcParams.update({'font.size': 10, 'figure.dpi': 160,'font.family': 'TeX Gyre Termes'})
# ----------------- helpers -----------------
def _pick_region_col(df):
    for c in ["Region", "Region_y", "Region_x", "region"]:
        if c in df.columns:
            return c
    raise KeyError("No region column found. Expected one of: Region, Region_y, Region_x, region.")

def _ensure_metric_crs(gdf, target_epsg: int = 3857):
    if gdf.crs is None:
        gdf = gdf.set_crs(4326, allow_override=True)
    if gdf.crs.to_epsg() != target_epsg:
        gdf = gdf.to_crs(target_epsg)
    return gdf

def _format_gw(val_mw: float) -> str:
    """Clean GW label rounded to 0.1 GW."""
    return f"{(val_mw/1000):.0f} GW"

# ----------------- plots -----------------
def plot_bubble_map(
    gdf: gpd.GeoDataFrame,
    size_col: str = "total_capacity",
    color_col: str = "pearson_C",
    region_label_col: str | None = None,
    annotate_top_n: int = 6,
    size_range_pts2: tuple[float, float] = (60, 1600),
    cmap_name: str = "seismic",
    clip_colors_to_quantiles: tuple[float, float] = (0.05, 0.95),
    title: str = "BC bubble map: area ∝ total capacity; color = complementarity (higher=more anti-correlation)",
    legend_gw_values: tuple[float, ...] = (1, 5, 10),   # <<< NEW: fixed GW legend
):
    """

    """
    if region_label_col is None:
        region_label_col = _pick_region_col(gdf)
    required = [size_col, color_col, region_label_col, "geometry"]
    missing = [c for c in required if c not in gdf.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")

    gg = _ensure_metric_crs(gdf.copy(), 3857)

    # positions (robust to concave polygons)
    pts = gg.geometry.representative_point()

    # bubble sizes (linear in area; guard zero-range)
    raw = pd.to_numeric(gg[size_col], errors="coerce").fillna(0).clip(lower=0)
    smin, smax = size_range_pts2
    if raw.max() > raw.min():
        s_scaled = smin + (raw - raw.min())/(raw.max() - raw.min())*(smax - smin)
    else:
        s_scaled = np.full(len(gg), (smin + smax)/2)

    # colors (quantile clip to reduce skew)
    cvals = pd.to_numeric(gg[color_col], errors="coerce")
    qlo, qhi = clip_colors_to_quantiles
    vmin = np.nanquantile(cvals, qlo) if np.isfinite(cvals).any() else np.nanmin(cvals)
    vmax = np.nanquantile(cvals, qhi) if np.isfinite(cvals).any() else np.nanmax(cvals)
    if not np.isfinite(vmin): vmin = np.nanmin(cvals)
    if not np.isfinite(vmax): vmax = np.nanmax(cvals)
    norm = Normalize(vmin=vmin, vmax=vmax)
    cmap = get_cmap(cmap_name)

    # figure
    fig, ax = plt.subplots(figsize=(8.6, 8.6), dpi=160)

    # soft background: filled polygons very light; thin boundary for context
    gg.plot(ax=ax, color="#F8F7F7", edgecolor="#3A3A3A", linewidth=0.15, zorder=1)

    # bubbles (white rim improves legibility)
    sc = ax.scatter(
        pts.x, pts.y,
        s=s_scaled,
        c=cvals, cmap=cmap, norm=norm,
        alpha=0.88, edgecolor="grey", linewidth=0.2, zorder=2
    )

    # top-N labels with halo
    if annotate_top_n and annotate_top_n > 0:
        top = gg.assign(_s=raw).sort_values("_s", ascending=False).head(annotate_top_n)
        tpts = top.geometry.representative_point()
        for (_, r), p in zip(top.iterrows(), tpts):
            ax.text(
                p.x, p.y, str(r[region_label_col]),
                fontsize=8, weight="bold",
                ha="left", va="center",
                bbox=dict(boxstyle="round,pad=0.15", facecolor="None", edgecolor="none", alpha=0.65),
                zorder=3
            )

    # colorbar
    cbar = plt.colorbar(ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=0.5, pad=0.01)

    # make it clean
    cbar.outline.set_visible(False)                       # remove outer box
    cbar.ax.tick_params(which="both", length=0, width=0)  # remove tick marks
    for spine in cbar.ax.spines.values():                 # extra safety if a style adds spines
        spine.set_visible(False)
    if hasattr(cbar, "solids"):                           # avoid faint band edges on some backends
        cbar.solids.set_edgecolor("face")

    cbar.set_label("Complementarity score (higher = more anti-correlation)")

# ---------------- legend: fixed 1, 5, 10 GW bubbles ----------------
    handles, labels = [], []
    for gw in legend_gw_values:
        mw = gw * 1000.0
        if raw.max() > raw.min():
            mw_clamped = np.clip(mw, raw.min(), raw.max())
            s = smin + (mw_clamped - raw.min())/(raw.max() - raw.min()) * (smax - smin)
        else:
            s = (smin + smax) / 2.0
        h = ax.scatter([], [], s=s, facecolors="none", edgecolors="black", linewidths=0.9)
        handles.append(h)
        labels.append(f"{gw:g} GW")
    leg = ax.legend(
        handles, labels, title="Total capacity",
        scatterpoints=1, frameon=False, loc="lower left", fontsize=10, title_fontsize=12
    )
    leg.get_frame().set_alpha(0.6)

    # ax.set_title(title, fontsize=11.5)
    ax.set_axis_off()
    fig.tight_layout()
    plt.show()


In [ ]:
gdf_out['total_capacity'] = gdf_out['potential_capacity_solar'] + gdf_out['potential_capacity_wind']

In [ ]:
gdf_out

In [ ]:
# Bubble map
plot_bubble_map(gdf_out, annotate_top_n=5,legend_gw_values=(0.5, 2, 8),cmap_name="YlOrRd")

In [ ]:
timeseries_solar_filtered=timeseries_solar[common_idx]
timeseries_wind_filtered=timeseries_wind[common_idx]
vis.plot_region_complementarity(timeseries_solar=timeseries_solar, 
                                timeseries_wind=timeseries_wind,
                                region_code=region_code,
                                RUN_ID=POLICY,
                                region='PeaceRiver',
                                # clusters=True,
                                # aggregate=True
                                )

- Supply curve filtered cells

In [ ]:
timeseries_solar_filtered=timeseries_solar[common_idx]
timeseries_wind_filtered=timeseries_wind[common_idx]
vis.plot_region_complementarity(timeseries_solar=timeseries_solar_filtered, 
                                timeseries_wind=timeseries_wind_filtered,
                                region_code=region_code,
                                RUN_ID=POLICY,
                                region='PeaceRiver',
                                clusters=True,
                                aggregate=True
                                )

- Clusters

In [ ]:
vis.plot_region_complementarity(timeseries_solar=timeseries_clusters_solar, 
                                timeseries_wind=timeseries_clusters_wind, 
                                region_code=region_code,
                                # region='PeaceRiver',
                                RUN_ID=POLICY,
                                # aggregate=True,
                                show=True)


In [ ]:
# # Calculate standard deviation of difference for each cluster vs PeaceRiver_1
# std_devs = timeseries_clusters_wind.subtract(timeseries_clusters_wind['PeaceRiver_1'], axis=0).std()

# # Find the cluster with the maximum deviation
# most_deviated_cluster = std_devs.idxmax()
# max_deviation = std_devs.max()

# print(f"Most deviated cluster: {most_deviated_cluster} (std={max_deviation:.4f})")


In [ ]:
# timeseries_clusters_wind[most_deviated_cluster].to_csv(f"../results/temp/timeseries_clusters_wind_{RUN_ID}_{most_deviated_cluster}.csv")

In [ ]:
# # Compute daily means for PeaceRiver_1 and most_deviated_cluster
# peace_daily = timeseries_clusters_wind['PeaceRiver_1'].resample('1D').mean()
# deviated_daily = timeseries_clusters_wind[most_deviated_cluster].resample('1D').mean()

# fig, ax = plt.subplots(figsize=(10, 4), dpi=300)
# ax.plot(peace_daily.index, peace_daily.values, label='PeaceRiver_1', color='navy', linewidth=2)
# ax.plot(deviated_daily.index, deviated_daily.values, label=most_deviated_cluster, color='orangered', linewidth=2)
# ax.set_title(f"Daily Mean Wind CF: PeaceRiver_1 vs {most_deviated_cluster}", fontsize=14)
# ax.set_ylabel("Capacity Factor")
# ax.set_xlabel("Date")
# ax.legend(frameon=False)
# ax.grid(alpha=0.3, linestyle='--')
# plt.tight_layout()
# # plt.show()

# Regional Outlines (not sensitive to Policies)

## Region Outline

In [ ]:
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 5), dpi=500)
region_names_anchors = (-0.01, 0.88)  # adjust legend placement

unique_regions = boundary_proj["Region_Number"].unique()
N = len(unique_regions)

# Option A: plt.get_cmap with N discrete colors
base_cmap = plt.get_cmap("tab20", N)
colors = [base_cmap(i) for i in range(N)]

# shuffle for randomness
np.random.shuffle(colors)

region_color_map = dict(zip(unique_regions, colors))
boundary_proj["color"] = boundary_proj["Region_Number"].map(region_color_map)

# --- Plot polygons with color fill ---
boundary_proj.plot(ax=ax,
                   linewidth=0.4,
                   alpha=0.6,
                   facecolor=boundary_proj["color"],
                   edgecolor="white",
                   )

ax.set_axis_off()

# --- Annotate region numbers with white halo ---
for idx, row in boundary_proj.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        x, y = row.geometry.centroid.x, row.geometry.centroid.y
        ax.text(
            x, y, str(row["Region_Number"]),
            ha="center", va="center",
            fontsize=10, fontweight="bold", color="black",
            path_effects=[pe.withStroke(linewidth=2, foreground="white",alpha=0.4)]
        )

# --- Legend ---
handles = [
    mpatches.Patch(facecolor=region_color_map[num], edgecolor="None",
                   label=f"{num} - {name}")
    for num, name in zip(region_mapping["Region_Number"], region_mapping["Region"])
]
ax.legend(handles=handles,
          bbox_to_anchor=region_names_anchors,
          loc="upper left", frameon=False, fontsize=6.5)

ax.set_facecolor("none")  # transparent background
plt.tight_layout()
plt.savefig(f"../vis/{country_kwd}/{region_code}/{region_code}_regions.png",
            bbox_inches="tight", transparent=True)


In [ ]:
if cells.crs!=boundary_proj.crs:
    cells = cells.to_crs(boundary_proj.crs)
fig, ax = plt.subplots(figsize=(7, 5),dpi=500, facecolor='none')
cells.boundary.plot(ax=ax, linewidth=0.5, color='k', alpha=1)
vis.add_compass_arrow_custom(ax,x=0.8,text_offset=0.03)
ax.set_axis_off()
# fig.patch.set_alpha(1)  # Make figure background transparent
# ax.set_facecolor('none')  # Make axis background transparent
plt.tight_layout()
plt.savefig(f"../vis/{country_kwd}/{region_code}/{region_code}_gridcells_outline.svg")
# plt.show()

# Grid

In [ ]:
lines=BC_dfs_all_runs[f'{POLICY}']['lines']
if lines.crs!=boundary_proj.crs:
    lines = lines.to_crs(boundary_proj.crs)

lines_cleaned=lines[lines['power']=='line']

In [ ]:
# for columns in lines.columns:
#     print(f"{columns}")

### lines

In [ ]:
def plot_grid_lines(
    region_code: str,
    region_name: str,
    lines: gpd.GeoDataFrame,
    boundary: gpd.GeoDataFrame,
    font_family: str = None,
    figsize: tuple = (10, 8),
    dpi=1000,
    save_to: str | Path = None,
    show: bool = True,
):
    """
    Plots transmission lines with binned voltage levels in a specified region.
    """
    lines = lines.copy() # avoid modifying original
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    fig.suptitle("Transmission Lines by Voltage Levels", fontsize=16, fontweight='bold')
    # plt.style.use(style_path)
    if font_family is not None:
        plt.rcParams['font.family'] = font_family

    boundary.plot(ax=ax, facecolor='White', edgecolor='black', linewidth=0.2, alpha=0.7)

    if 'voltage' in lines.columns:
        # Convert to numeric
        lines['voltage_kv'] = pd.to_numeric(lines['voltage'], errors='coerce') / 1000

        # Define voltage bins
        bins = [0, 12, 25, 132, 220, float("inf")]
        labels = ["<12 kV", "12–25 kV", "25–132 kV", "132–220 kV", "≥220 kV"]
        lines['voltage_class'] = pd.cut(lines['voltage_kv'], bins=bins, labels=labels, right=False)

        # Color map (enough distinct colors)
        cmap = plt.colormaps.get_cmap('tab10')
        colors = [cmap(i) for i in range(len(labels))]
        color_map = {label: colors[i] for i, label in enumerate(labels)}


        # Plot by class
        for label in labels:
            mask = lines['voltage_class'] == label
            if mask.any():
                lines[mask].plot(ax=ax, color=color_map[label], linewidth=1, alpha=0.8)

        # Legend
        legend_patches = [mpatches.Patch(color=color_map[label], label=label) for label in labels if label in lines['voltage_class'].unique()]
        ax.legend(handles=legend_patches, frameon=False, fontsize=11, loc='upper right')

    else:
        lines.plot(ax=ax, color='blue', linewidth=1,alpha=0.7)

    ax.set_axis_off()
    plt.tight_layout()

    if save_to is None:
        save_to = Path("vis") / region_code / "network"
    else:
        save_to = Path(save_to)
    
    save_to.mkdir(parents=True, exist_ok=True)
    save_to_file = save_to / f"transmission_lines_{region_code}.svg"
    plt.savefig(save_to_file, bbox_inches='tight', dpi=300,transparent=True)
    
    utils.print_update(level=2, message=f"Transmission Lines for {region_name} saved to {save_to_file}")
    if show:
        plt.show()
    return lines

In [ ]:
lines_new=plot_grid_lines(
    region_code=region_code,
    region_name=region_name,
    lines=lines_cleaned,
    boundary=boundary_proj,
    figsize=(7, 5),
    dpi=1000,
    save_to=f"../vis/{country_kwd}/{region_code}/",
    show=True
)

### Grid Proximity

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Define custom bins and labels for solar and wind capacity
bins = [0, 5, 20, 50, 100, float('inf')]  # Custom ranges
labels = ['<5','5-20', '20-50', '50-100', '>100']  # Labels for legend

if cells_scenario.crs!=boundary_proj.crs:
    cells_proj= cells.to_crs(boundary_proj.crs)
    
# Categorize potential_capacity_solar and potential_capacity_wind into bins
cells_proj['station_distance_category'] = pd.cut(cells['nearest_station_distance_km'], bins=bins, labels=labels, include_lowest=True)


# Create figure and axes for side-by-side plotting
fig, (ax) = plt.subplots(figsize=(7, 5),dpi=1000)
fig.suptitle("Proximity to Existing Grid Nodes", fontsize=16, fontweight='bold',y=0.95)
ax.set_axis_off()

# Shadow effect offset
shadow_offset = 0.002

# Plot solar map on ax1
# Add shadow effect for solar map
# boundary_proj.geometry = boundary_proj.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
boundary_proj.plot(ax=ax, facecolor='none', edgecolor='gray', linewidth=0.2, alpha=0.4)  # Shadow layer
# boundary_proj.geometry = boundary_proj.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot solar cells
cells_proj.plot(
    column='station_distance_category',
    ax=ax,
    cmap='copper',
    legend=True,
    linewidth=0.1,
    legend_kwds={
        'title': "Nearest sub-stations (km)",
        'title_fontsize': 11,
        'loc': 'upper right',
        'bbox_to_anchor': (0.38, 0.35),
        'fontsize':10.5,
        'frameon': False
    }
)

# Plot actual boundary for solar map
boundary_proj.plot(ax=ax, facecolor='None', edgecolor='black', linewidth=0.5, alpha=0.9)

# Adjust layout for cleaner appearance
# fig.patch.set_alpha(0)  # Make figure background transparent
plt.tight_layout()
save_to = Path(f"../vis/{country_kwd}/{region_code}/Resources_proximity_to_grid_{region_code}.svg",transparent=True)
plt.savefig(save_to, bbox_inches='tight')
plt.savefig(f"../docs/source/_static/Resources_proximity_to_grid_{region_code}.jpg", bbox_inches='tight')

# Maps

* Individual Maps

In [ ]:
# vis.get_data_in_map_plot(cells=cells_baseline_proj, 
#                 resource_type='solar', 
#                 datafield='CAPACITY',
#                 title="Solar Resources",
#                 # ax=ax1,
#                 # font_family='sans-serif',
#                 show=False)

In [ ]:
# vis.get_data_in_map_plot(cells, 
#                 resource_type='wind', 
#                   datafield='CF',
#                 title=f"Wind Resources for {region_name}",
#                 # ax=ax1, 
#                 show=False)

* Combined Map

In [ ]:
supported_datafields=['CF','CAPACITY','SCORE'] # that's what I configured in the plot func

In [ ]:
import matplotlib.pyplot as plt

for datafield in supported_datafields:
    for policy in POLICYs:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10,5), dpi=1000)

        cells= BC_dfs_all_runs[policy]['cells']
        if cells.crs!=boundary_proj.crs:
            cells_proj= cells.to_crs(boundary_proj.crs)
        # fig.suptitle(f"Renewable Resources in {region_name} ({policy})", fontsize=16, fontweight='bold')
        vis.get_data_in_map_plot(cells_proj, 
                        resource_type='solar',
                        datafield=datafield,
                        ax=ax1, 
                        score_threshold=600,
                        show=False)
        vis.get_data_in_map_plot(cells_proj, 
                        resource_type='wind',
                        datafield=datafield,
                        ax=ax2, 
                        score_threshold=600,
                        show=False)
        # vis.add_compass_arrow_custom(ax1, text_offset=0.04)
        vis.add_compass_arrow_custom(ax2, text_offset=0.04)
        plt.tight_layout()
        save_to_root= BASELINE_vis_save_to if policy=='BASELINE' else POLICY_vis_save_to
        
        plt.savefig(f"{save_to_root}/Resources_combined_{datafield}.svg", bbox_inches='tight', transparent=False)
        
        if policy=='BASELINE': 
            plt.savefig(f"../docs/source/_static/Resources_combined_{datafield}.png", bbox_inches='tight', transparent=False)

- Score Mapping (bins)

In [ ]:
cells_proj_solar_clean=cells_proj[cells_proj['lcoe_solar']<=150]
cells_proj_wind_clean=cells_proj[cells_proj['lcoe_wind']<150]

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# Ensure 'Region' is in the columns for both boundary and cells
# if 'Region' not in boundary.columns:
#     boundary = boundary.reset_index(inplace=True)

# # Assign a number to each region
# boundary['Region_Number'] = range(1, len(boundary) + 1)

# Define custom bins and labels for solar and wind capacity
solar_bins = [30, 40, 50, 60, 70,80, 90,100, float('inf')]  # Custom ranges
solar_labels = ['<30','30-40','40-50','50-60','60-70', '70-80', '80-90','>100']  # Labels for legend
# Define custom bins and labels for solar and wind capacity
wind_bins = [30, 40, 50, 60, 70,80, 90,100, float('inf')] 
wind_labels =['<30','30-40','40-50','50-60','60-70', '70-80', '80-90','>100']  # Labels for legend

# Categorize potential_capacity_solar and potential_capacity_wind into bins
cells_proj_solar_clean['solar_category'] = pd.cut(cells_proj_solar_clean['lcoe_solar'], bins=solar_bins, labels=solar_labels, include_lowest=True)
cells_proj_wind_clean['wind_category'] = pd.cut(cells_proj_wind_clean['lcoe_wind'], bins=wind_bins, labels=wind_labels, include_lowest=True)

# Create figure and axes for side-by-side plotting
fig, (ax1, ax2) = plt.subplots(figsize=(9, 4), ncols=2,dpi=1000)
fig.suptitle("Relative Cost Scoring ($/MWh)", fontsize=14, fontweight='bold')
# Set axis off for both subplots
ax1.set_axis_off()
ax2.set_axis_off()

# Shadow effect offset
# shadow_offset = 0.001

# Plot solar map on ax1
# Add shadow effect for solar map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax1, facecolor='none', edgecolor='gray', linewidth=1.2, alpha=0.3)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot solar cells
cells_proj_solar_clean.plot(column='solar_category', ax=ax1, cmap='YlOrRd', legend=False, edgecolor='white',linewidth=0.2,alpha=1,
        #    legend_kwds={'title': "Solar",'title_fontsize':14, 'bbox_to_anchor':(legend_x_ax_offset,legend_y_ax_offset),'fontsize':14,'frameon': False}
           )
cells_proj_wind_clean.plot(ax=ax1,color='grey', alpha=0.2,zorder=2)
# # Plot actual boundary for solar map
# boundary.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.9)

""" 
# Annotate region numbers for solar map
for idx, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax1.annotate(f"{row['Region_Number']}", 
                 xy=(centroid.x, centroid.y), 
                 ha='center', va='center',
                 fontsize=7, color='black',
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
"""
# Plot wind map on ax2
# Add shadow effect for wind map
# boundary.geometry = boundary.geometry.translate(xoff=shadow_offset, yoff=-shadow_offset)
# boundary.plot(ax=ax2, color='None', edgecolor='k', linewidth=0.2, alpha=0.7)  # Shadow layer
# boundary.geometry = boundary.geometry.translate(xoff=-shadow_offset, yoff=shadow_offset)

# Plot wind cells
cells_proj_wind_clean.plot(column='wind_category', ax=ax2, cmap='BuPu', legend=False, edgecolor='white',linewidth=0.2,alpha=1,
        #    legend_kwds={'title': "Wind", 'title_fontsize':14, 'bbox_to_anchor':(legend_x_ax_offset,legend_y_ax_offset),'fontsize':14,'frameon': False}
           )
cells_proj.plot(ax=ax2,color='grey', alpha=0.2,zorder=2)

# Plot actual boundary for wind map
# boundary.plot(ax=ax2, facecolor='none', edgecolor='black', linewidth=0.2, alpha=0.9)
"""
# Annotate region numbers for wind map
for idx, row in boundary.iterrows():
    centroid = row.geometry.centroid
    ax2.annotate(f"{row['Region_Number']}", 
                 xy=(centroid.x, centroid.y), 
                 ha='center', va='center',
                 fontsize=8, color='black',
                 bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, boxstyle='round,pad=0.2'))
"""
# Adjust layout for cleaner appearance
fig.patch.set_alpha(0)  # Make figure background transparent
# Add annotation to the figure
fig.text(0.5, 0.01, 
         "Note: The Scoring is calculated to reflect Dollar investment required to get an unit of Energy yield (MWh). "
         "\nTo reflect market competitiveness and incentives, the Score ($/MWh) needs financial adjustment factors to be considered on top of it.",
         ha='center', va='center', fontsize=10, color='k', bbox=dict(facecolor='None', edgecolor='k',linewidth=0.2,boxstyle='round,pad=0.5'))
plt.tight_layout()

# Show the side-by-side plot
# vis.add_compass_arrow_custom(ax1,text_offset=0.03)
vis.add_compass_arrow_custom(ax2,x=0.78,text_offset=0.04)
plt.savefig(f'{POLICY_vis_save_to}/solar_wind_score_map.jpg')

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

# --- Solar Plot ---
fig_solar, ax_solar = plt.subplots(figsize=(7, 7), dpi=1000)
# fig_solar.suptitle("Solar Relative Cost Score ($/MWh)", fontsize=14, fontweight='bold')

ax_solar.set_axis_off()
fig_solar.patch.set_alpha(0)

# Plot solar cells
cells_proj_solar_clean.plot(
    column='solar_category',
    ax=ax_solar,
    cmap='YlOrRd',
    legend=False,
    edgecolor='white',
    linewidth=0.2,
    alpha=1
)
# Overlay wind cells lightly for context
cells_proj.plot(ax=ax_solar, color='grey',edgecolor='white',linewidth=1, alpha=0.2, zorder=2)

# Compass arrow
# vis.add_compass_arrow_custom(ax_solar, x=0.78, text_offset=0.04)

plt.tight_layout()
plt.savefig(f"{POLICY_vis_save_to}/cost_map_solar.svg",transparent=True)

In [ ]:
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# Define colormaps
solar_cmap = cm.get_cmap('YlOrRd', len(solar_labels))
wind_cmap = cm.get_cmap('BuPu', len(wind_labels))

# Generate colors for each bin
solar_colors = [mcolors.rgb2hex(solar_cmap(i)) for i in range(len(solar_labels))]
wind_colors = [mcolors.rgb2hex(wind_cmap(i)) for i in range(len(wind_labels))]

# Aggregate potential capacity for each bin
solar_capacity = (
    cells_proj_solar_clean.groupby('solar_category')['potential_capacity_solar']
    .sum().div(1e3)
    .reindex(solar_labels, fill_value=0)
)
wind_capacity = (
    cells_proj_wind_clean.groupby('wind_category')['potential_capacity_wind']
    .sum().div(1e3)
    .reindex(wind_labels, fill_value=0)
)

# --- Drop bins with 0 capacity ---
solar_capacity = solar_capacity[solar_capacity > 0]
wind_capacity = wind_capacity[wind_capacity > 0]

solar_colors_filtered = [solar_colors[solar_labels.index(lbl)] for lbl in solar_capacity.index]
wind_colors_filtered = [wind_colors[wind_labels.index(lbl)] for lbl in wind_capacity.index]

In [ ]:
### Solar Plot (Horizontal, lowest cost on top) ###
fig1, ax1 = plt.subplots(figsize=(6, 3), dpi=500, constrained_layout=True)
fig1.patch.set_alpha(0)
ax1.set_facecolor('none')

# Horizontal bar plot
ax1.barh(solar_capacity.index, solar_capacity.values,
         color=solar_colors_filtered, edgecolor='none')

ax1.set_xlabel('Solar Potential (GW)', fontsize=14, weight='bold')
ax1.set_ylabel('Relative cost score ($/MWh)', fontsize=14)

# Put lowest scores at the top
ax1.invert_yaxis()

ax1.grid(False)

for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.tick_params(bottom=True, left=True, labelsize=14)

plt.savefig(f'{POLICY_vis_save_to}/cost_barchart_solar.svg', transparent=True)

In [ ]:

# --- Wind Plot ---
fig_wind, ax_wind = plt.subplots(figsize=(7, 7), dpi=1000)
# fig_wind.suptitle("Wind Relative Cost Score ($/MWh)", fontsize=14, fontweight='bold')

ax_wind.set_axis_off()
fig_wind.patch.set_alpha(0)

# Plot wind cells
cells_proj_wind_clean.plot(
    column='wind_category',
    ax=ax_wind,
    cmap='BuPu',
    legend=False,
    edgecolor='white',
    linewidth=0.2,
    alpha=1
)
# Overlay solar cells lightly for context
cells_proj.plot(ax=ax_wind, color='grey',edgecolor='white',linewidth=1, alpha=0.2, zorder=2)
# Compass arrow
# vis.add_compass_arrow_custom(ax_wind, x=0.78, text_offset=0.04)

plt.tight_layout()
plt.savefig(f"{POLICY_vis_save_to}/cost_map_wind.svg",transparent=True)

In [ ]:
### Wind Plot (Horizontal, lowest cost on top) ###
fig2, ax2 = plt.subplots(figsize=(6, 3), dpi=500, constrained_layout=True)
fig2.patch.set_alpha(0)
ax2.set_facecolor('none')

# Horizontal bar plot
ax2.barh(wind_capacity.index, wind_capacity.values,
         color=wind_colors_filtered, edgecolor='none')

ax2.set_xlabel('Wind Potential (GW)', fontsize=14, weight='bold')
ax2.set_ylabel('Relative cost score ($/MWh)', fontsize=14)

# Put lowest scores at the top
ax2.invert_yaxis()

ax2.grid(False)

for spine in ax2.spines.values():
    spine.set_visible(False)
ax2.tick_params(bottom=True, left=True, labelsize=14)

plt.savefig(f'{POLICY_vis_save_to}/cost_barchart_wind.svg', transparent=True)

In [ ]:
# Example of correct argument dictionary (if you want to use kwargs)
scatter_plot_args = {
    "solar_clusters": clusters_solar_f,
    "wind_clusters":  clusters_wind_f,
    "bubbles_GW": [1,5,10], # bubble sizes in GW
    "bubbles_scale": 0.03, #value 0.01 for 100 times smaller than the original scale 
    "lcoe_threshold": 120, # LCOE threshold in $/MWh
    "figsize": (7, 3.5), # Adjusted figure size for publication quality
    "dpi":1000,
    "save_to_root": f"{POLICY_vis_save_to}",
}

# Call the function using the argument dictionary
vis.plot_resources_scatter_metric_combined(**scatter_plot_args)

## CF checks

In [ ]:
gwa_country_code=cfg_policy.get('region_mapping').get(region_code).get('GWA_country_code')
utils.print_banner(f"GWA Country Code Selected: {gwa_country_code}")

### Wind

In [ ]:
# Get total bounds from boundary GeoDataFrame
minx, miny, maxx, maxy = boundary.total_bounds

# Create bounding_box_dict with correct keys for downstream use
bounding_box_dict = {
    "minx": float(minx),
    "miny": float(miny),
    "maxx": float(maxx),
    "maxy": float(maxy)
}


In [ ]:
import rioxarray as rxr

raster_path=f'../data/downloaded_data/GWA/{gwa_country_code}_capacity-factor_IEC3.tif'
gwa_raster_data = (
        rxr.open_rasterio(raster_path)
        .rio.clip_box(**bounding_box_dict)
        .rename('CF_IEC3')
        .drop_vars(['band', 'spatial_ref'])
        .isel(band=1 if '*Class*' in 'CF_IEC3' else 0)  # 'IEC_Class_ExLoads' data is in band 1
    )


In [ ]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(3.5, 2.5),dpi=500)
# gwa_raster_data.plot(ax=ax, cmap='BuPu', add_colorbar=True)
# boundary.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=0.5)
# ax.set_title("GWA CF-IEC3 Reference (High-res)")
# ax.axis('off')
# plt.savefig(f"../vis/{region_code}/GWA_CF_IEC3.png", bbox_inches='tight', transparent=False)

In [ ]:
# vis.get_CF_wind_check_plot(cells, 
#                        gwa_raster_data,
#                        boundary,
#                        region_code,
#                        region_name,
#                        ['CF_IEC3', 'wind_CF_mean'],
#                        figure_height=7,)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean and minimal style
sns.set_style("white")

# Initialize the figure
plt.figure(figsize=(7.5, 2.5),dpi=1000)
plt.style.use('../RES/visual_styles/elsevier.mplstyle')

# Create the boxplot
ax = sns.boxplot(
    data=cells[['CF_IEC2', 'CF_IEC3', 'wind_CF_mean']],
    palette="Paired",
    linewidth=0.2,
    width=0.6
)

# Set title and labels
ax.set_title('Capacity Factor Distribution Comparison', weight='semibold', pad=12)
ax.set_ylabel('Capacity Factor')
ax.set_xlabel('')

# Tweak tick formatting
ax.tick_params(axis='x')
ax.tick_params(axis='y')

# Remove all spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Add horizontal grid lines
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.xaxis.grid(False)

plt.figtext(
    0.01, -0.08,
    "*'CF_IEC2, CF_IEC3: Average CF for ERA5 Cells, calculated from high-resolution yearly average CF from GWA for IEC Class 2 and 3 turbines.\n"
    "*wind_CF_mean: Average CF for ERA5 Cells, calculated from the ERA5 windspeed (rescaled with GWA) time series ",
    ha='left', fontsize=7, style='normal', fontweight='normal', color='gray',wrap=True
)

plt.tight_layout()
plt.savefig(f"../vis/{country_kwd}/{region_code}/CF_distribution_comparison.svg", bbox_inches='tight', transparent=False)

# GWA raster vs ERA5

* Plot wind CF raster (benchmark) vs Calculated CF for ERA5 Cells

In [ ]:
gwa_country_code=cfg_policy.get('region_mapping').get(region_code).get('GWA_country_code')
utils.print_banner(f"GWA Country Code Selected: {gwa_country_code}")

In [ ]:
import rioxarray as rxr

raster_path=f'../data/downloaded_data/GWA/{gwa_country_code}_capacity-factor_IEC3.tif'
gwa_raster_data = (
        rxr.open_rasterio(raster_path)
        .rio.clip_box(**bounding_box_dict)
        .rename('CF_IEC3')
        .isel(band=1 if '*Class*' in 'CF_IEC3' else 0)  # 'IEC_Class_ExLoads' data is in band 1
        # .drop_vars(['band', 'spatial_ref']) # removes CRS info
    )

In [ ]:
# import matplotlib.pyplot as plt

# fig, ax = plt.subplots(figsize=(3.5, 2.5),dpi=500)
# gwa_raster_data.plot(ax=ax, cmap='BuPu', add_colorbar=True)
# boundary.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=0.5)
# ax.set_title("GWA CF-IEC3 Reference (High-res)")
# ax.axis('off')
# plt.savefig(f"../vis/{region_code}/GWA_CF_IEC3.png", bbox_inches='tight', transparent=False)

In [ ]:
if gwa_raster_data.rio.crs != CRS_m:
    gwa_raster_plot = gwa_raster_data.rio.reproject(CRS_m)   # <- fixed typo
else:
    gwa_raster_plot = gwa_raster_data
    
if cells_scenario.crs != CRS_m:
    cells_scenario_proj=cells_scenario.to_crs(CRS_m)
else:
    cells_scenario_proj=cells_scenario

In [ ]:
# vis.get_CF_wind_check_plot(cells_baseline, 
#                        gwa_raster_plot,
#                        boundary_proj,
#                        region_code,
#                        region_name,
#                        ['CF_IEC3', 'wind_CF_mean'],
#                        font_family='sans-serif',
#                        figure_height=4,
#                        save_to=f"{BASELINE_vis_save_to}/GWA_CF_IEC3_vs_cells_wind_CF_mean_{region_code}.png")

In [ ]:
vis.get_CF_wind_check_plot(cells_scenario_proj, 
                       gwa_raster_plot,
                       boundary_proj,
                       region_code,
                       region_name,
                       ['CF_IEC3', 'wind_CF_mean'],
                       font_family='sans-serif',
                       figure_height=5,
                       save_to=f"{POLICY_vis_save_to}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a clean and minimal style
sns.set_style("white")

# Initialize the figure
plt.figure(figsize=(7.5, 2.5),dpi=1000)
plt.style.use('../RES/visual_styles/elsevier.mplstyle')

# Create the boxplot
ax = sns.boxplot(
    data=cells_scenario[['CF_IEC2', 'CF_IEC3', 'wind_CF_mean']],
    palette="Paired",
    linewidth=0.2,
    width=0.6
)

# Set title and labels
ax.set_title('Capacity Factor Distribution Comparison', weight='semibold', pad=12)
ax.set_ylabel('Capacity Factor')
ax.set_xlabel('')

# Tweak tick formatting
ax.tick_params(axis='x')
ax.tick_params(axis='y')

# Remove all spines
for spine in ax.spines.values():
    spine.set_visible(False)

# Add horizontal grid lines
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.xaxis.grid(False)

plt.figtext(
    0.01, -0.08,
    "*'CF_IEC2, CF_IEC3: Average CF for ERA5 Cells, calculated from high-resolution yearly average CF from GWA for IEC Class 2 and 3 turbines.\n"
    "*wind_CF_mean: Average CF for ERA5 Cells, calculated from the ERA5 windspeed (rescaled with GWA) time series ",
    ha='left', fontsize=7, style='normal', fontweight='normal', color='gray',wrap=True
)

plt.tight_layout()
plt.savefig('../docs/source/_static/CF_distribution_comparison.png', bbox_inches="tight")
plt.savefig(f"{POLICY_vis_save_to}/CF_distribution_comparison.png", bbox_inches='tight', transparent=False)